machine learning model ##

In [ ]:
# ============================================================
# STEP 6A — UNSUPERVISED ML INPUT VERIFICATION
# ============================================================

import os
import hashlib
import json
import pandas as pd
import numpy as np

SOURCE = "/content/session_behavioral_features_v4/session_behavioral_features_v4.csv"

EXPECTED_SHA = (
    "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"
)

EXPECTED_ROWS = 11880

# Labels that MUST NOT enter unsupervised ML
FORBIDDEN_LABELS = {
    "attack_type",
    "attack_stage",
    "severity",
    "mitre_technique"
}

# ------------------------------------------------------------
# 1. SOURCE CHECK
# ------------------------------------------------------------

print("=" * 70)
print("STEP 6A — UNSUPERVISED ML INPUT VERIFICATION")
print("=" * 70)

if not os.path.exists(SOURCE):
    raise FileNotFoundError(
        f"STOP — verified Step 5 feature dataset not found:\n{SOURCE}"
    )

print("\n[1] SOURCE")
print(f"Exists: PASS")
print(f"Path: {SOURCE}")
print(f"Size: {os.path.getsize(SOURCE):,} bytes")


# ------------------------------------------------------------
# 2. SHA-256
# ------------------------------------------------------------

print("\n[2] SHA-256")
print("Calculating SHA-256...")

sha256 = hashlib.sha256()

with open(SOURCE, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

ACTUAL_SHA = sha256.hexdigest()

print(f"Expected: {EXPECTED_SHA}")
print(f"Actual:   {ACTUAL_SHA}")

if ACTUAL_SHA != EXPECTED_SHA:
    raise ValueError(
        "\nSTOP — SHA-256 MISMATCH.\n"
        "This is NOT the verified Step 5 feature checkpoint."
    )

print("Verification: PASS")


# ------------------------------------------------------------
# 3. LOAD
# ------------------------------------------------------------

print("\n[3] LOADING VERIFIED FEATURE DATASET")

df = pd.read_csv(SOURCE)

print(f"Shape: {df.shape}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

if len(df) != EXPECTED_ROWS:
    raise ValueError(
        f"STOP — unexpected row count: {len(df)} "
        f"(expected {EXPECTED_ROWS})"
    )

print("Row count: PASS")


# ------------------------------------------------------------
# 4. SESSION ID
# ------------------------------------------------------------

print("\n[4] SESSION ID VALIDATION")

SESSION_ID = "reconstructed_session_id"

if SESSION_ID not in df.columns:
    raise ValueError(
        f"STOP — missing required session ID: {SESSION_ID}"
    )

missing_ids = df[SESSION_ID].isna().sum()
duplicate_ids = df[SESSION_ID].duplicated().sum()
unique_ids = df[SESSION_ID].nunique()

print(f"Missing session IDs:   {missing_ids}")
print(f"Duplicate session IDs: {duplicate_ids}")
print(f"Unique session IDs:    {unique_ids}")

if missing_ids != 0:
    raise ValueError("STOP — missing session IDs detected.")

if duplicate_ids != 0:
    raise ValueError("STOP — duplicate session IDs detected.")

if unique_ids != EXPECTED_ROWS:
    raise ValueError(
        f"STOP — unexpected unique session count: {unique_ids}"
    )

print("Session integrity: PASS")


# ------------------------------------------------------------
# 5. LABEL LEAKAGE CHECK
# ------------------------------------------------------------

print("\n[5] LABEL LEAKAGE CHECK")

present_labels = sorted(
    FORBIDDEN_LABELS.intersection(df.columns)
)

print(f"Forbidden label columns present: {present_labels}")

if present_labels:
    raise ValueError(
        "STOP — label columns detected in ML input: "
        + str(present_labels)
    )

print("Label leakage: NO")


# ------------------------------------------------------------
# 6. FEATURE / ID SEPARATION
# ------------------------------------------------------------

print("\n[6] FEATURE / ID SEPARATION")

feature_columns = [
    c for c in df.columns
    if c != SESSION_ID
]

print(f"Total columns:   {len(df.columns)}")
print(f"Session ID:      {SESSION_ID}")
print(f"ML input columns: {len(feature_columns)}")

if not feature_columns:
    raise ValueError("STOP — no ML feature columns available.")

print("Feature selection: PASS")


# ------------------------------------------------------------
# 7. NUMERIC FEATURE INVENTORY
# ------------------------------------------------------------

print("\n[7] NUMERIC FEATURE INVENTORY")

numeric_columns = df[feature_columns].select_dtypes(
    include=[np.number]
).columns.tolist()

non_numeric_columns = [
    c for c in feature_columns
    if c not in numeric_columns
]

print(f"Numeric features:     {len(numeric_columns)}")
print(f"Non-numeric columns:  {len(non_numeric_columns)}")

if non_numeric_columns:
    print("\nNon-numeric columns:")
    for c in non_numeric_columns:
        print(f" - {c}")

print("\nNumeric features will be used for clustering.")
print("Categorical/time columns will NOT be silently encoded here.")


# ------------------------------------------------------------
# 8. FINITE VALUE CHECK
# ------------------------------------------------------------

print("\n[8] NUMERIC FINITE CHECK")

nonfinite = {}

for c in numeric_columns:
    values = pd.to_numeric(df[c], errors="coerce")

    count = (~np.isfinite(values)).sum()

    if count > 0:
        nonfinite[c] = int(count)

print(f"Non-finite numeric columns: {nonfinite}")

# Documented IP reputation missingness is expected from Step 5.
DOCUMENTED_MISSING = {
    "ip_reputation_mean",
    "ip_reputation_min",
    "ip_reputation_max",
    "ip_reputation_range",
    "ip_reputation_abs_mean",
}

unexpected_nonfinite = {
    c: n for c, n in nonfinite.items()
    if c not in DOCUMENTED_MISSING
}

print(f"Unexpected non-finite columns: {unexpected_nonfinite}")

if unexpected_nonfinite:
    raise ValueError(
        "STOP — unexpected non-finite values detected."
    )

print("Numeric sanity: PASS")


# ------------------------------------------------------------
# 9. SOURCE PROTECTION
# ------------------------------------------------------------

print("\n[9] SOURCE PROTECTION")

sha_after = hashlib.sha256()

with open(SOURCE, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha_after.update(chunk)

AFTER_SHA = sha_after.hexdigest()

print(f"SHA before: {ACTUAL_SHA}")
print(f"SHA after:  {AFTER_SHA}")

if AFTER_SHA != ACTUAL_SHA:
    raise ValueError(
        "STOP — source dataset changed during verification."
    )

print("Source unchanged: PASS")


# ------------------------------------------------------------
# 10. AUDIT SUMMARY
# ------------------------------------------------------------

audit = {
    "step": "6A",
    "status": "PASS",
    "source": SOURCE,
    "source_sha256": ACTUAL_SHA,
    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "unique_sessions": int(unique_ids),
    "missing_session_ids": int(missing_ids),
    "duplicate_session_ids": int(duplicate_ids),
    "forbidden_labels_present": present_labels,
    "numeric_feature_count": int(len(numeric_columns)),
    "non_numeric_columns": non_numeric_columns,
    "documented_nonfinite": {
        k: int(v) for k, v in nonfinite.items()
        if k in DOCUMENTED_MISSING
    },
    "unexpected_nonfinite": unexpected_nonfinite,
    "source_modified": False,
}

AUDIT_DIR = "/content/step6_unsupervised_ml"
os.makedirs(AUDIT_DIR, exist_ok=True)

AUDIT_PATH = os.path.join(
    AUDIT_DIR,
    "step6a_ml_input_verification_audit.json"
)

with open(AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2)

print("\n[10] AUDIT")
print(f"Audit: {AUDIT_PATH}")

print("\n" + "=" * 70)
print("STEP 6A STATUS: PASS")
print("=" * 70)

print("\nSAFE TO CONTINUE TO STEP 6B — ML PREPROCESSING")

STEP 6A — UNSUPERVISED ML INPUT VERIFICATION

[1] SOURCE
Exists: PASS
Path: /content/session_behavioral_features_v4/session_behavioral_features_v4.csv
Size: 4,814,141 bytes

[2] SHA-256
Calculating SHA-256...
Expected: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Verification: PASS

[3] LOADING VERIFIED FEATURE DATASET
Shape: (11880, 56)
Rows: 11,880
Columns: 56
Row count: PASS

[4] SESSION ID VALIDATION
Missing session IDs:   0
Duplicate session IDs: 0
Unique session IDs:    11880
Session integrity: PASS

[5] LABEL LEAKAGE CHECK
Forbidden label columns present: []
Label leakage: NO

[6] FEATURE / ID SEPARATION
Total columns:   56
Session ID:      reconstructed_session_id
ML input columns: 55
Feature selection: PASS

[7] NUMERIC FEATURE INVENTORY
Numeric features:     53
Non-numeric columns:  2

Non-numeric columns:
 - session_start_time
 - session_end_time

Numeric features will be used for 

In [ ]:
# ============================================================
# STEP 6B — UNSUPERVISED ML PREPROCESSING
# ============================================================

import os
import json
import hashlib
import shutil
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

SOURCE = "/content/session_behavioral_features_v4/session_behavioral_features_v4.csv"
SOURCE_SHA = "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"

OUTPUT_DIR = "/content/step6_unsupervised_ml/step6b_preprocessing"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(
    OUTPUT_DIR,
    "session_behavioral_features_v4_ml_ready.csv"
)

SCALER_PATH = os.path.join(
    OUTPUT_DIR,
    "standard_scaler_params.json"
)

AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "step6b_preprocessing_audit.json"
)

SHA_PATH = os.path.join(
    OUTPUT_DIR,
    "session_behavioral_features_v4_ml_ready.sha256"
)

# ------------------------------------------------------------
# SHA256
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

print("=" * 70)
print("STEP 6B — UNSUPERVISED ML PREPROCESSING")
print("=" * 70)

# ------------------------------------------------------------
# 1. SOURCE CHECK
# ------------------------------------------------------------

if not os.path.exists(SOURCE):
    raise FileNotFoundError(
        f"STOP — verified Step 6A source not found:\n{SOURCE}"
    )

actual_source_sha = sha256_file(SOURCE)

print("\n[1] SOURCE")
print(f"Path: {SOURCE}")
print(f"Exists: PASS")
print(f"Expected SHA: {SOURCE_SHA}")
print(f"Actual SHA:   {actual_source_sha}")

if actual_source_sha != SOURCE_SHA:
    raise ValueError(
        "STOP — SOURCE SHA-256 mismatch.\n"
        "Do NOT continue."
    )

print("Source SHA verification: PASS")

# ------------------------------------------------------------
# 2. LOAD
# ------------------------------------------------------------

df = pd.read_csv(SOURCE)

print("\n[2] DATASET")
print(f"Shape: {df.shape}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

if len(df) != 11880:
    raise ValueError(
        f"STOP — unexpected row count: {len(df)}"
    )

# ------------------------------------------------------------
# 3. SESSION ID
# ------------------------------------------------------------

SESSION_ID = "reconstructed_session_id"

if SESSION_ID not in df.columns:
    raise ValueError(
        f"STOP — missing {SESSION_ID}"
    )

missing_ids = int(df[SESSION_ID].isna().sum())
duplicate_ids = int(df[SESSION_ID].duplicated().sum())
unique_ids = int(df[SESSION_ID].nunique())

print("\n[3] SESSION INTEGRITY")
print(f"Missing session IDs:   {missing_ids}")
print(f"Duplicate session IDs: {duplicate_ids}")
print(f"Unique session IDs:    {unique_ids}")

if missing_ids != 0 or duplicate_ids != 0 or unique_ids != 11880:
    raise ValueError(
        "STOP — session integrity failure."
    )

print("Session integrity: PASS")

# ------------------------------------------------------------
# 4. LEAKAGE CHECK
# ------------------------------------------------------------

FORBIDDEN = {
    "attack_type",
    "attack_stage",
    "severity",
    "mitre_technique",
    "country",
    "country_code",
    "original_country",
    "original_country_code",
}

leakage_present = sorted(
    c for c in df.columns if c in FORBIDDEN
)

print("\n[4] LABEL LEAKAGE")
print(f"Forbidden columns present: {leakage_present}")

if leakage_present:
    raise ValueError(
        f"STOP — forbidden label columns detected: {leakage_present}"
    )

print("Label leakage: NO")

# ------------------------------------------------------------
# 5. FEATURE SELECTION
# ------------------------------------------------------------

EXCLUDED_FOR_CLUSTERING = {
    SESSION_ID,
    "session_start_time",
    "session_end_time",
}

feature_columns = [
    c for c in df.columns
    if c not in EXCLUDED_FOR_CLUSTERING
]

# Keep only numeric features.
numeric_features = [
    c for c in feature_columns
    if pd.api.types.is_numeric_dtype(df[c])
]

non_numeric_features = [
    c for c in feature_columns
    if c not in numeric_features
]

print("\n[5] FEATURE SELECTION")
print(f"Total source columns:       {len(df.columns)}")
print(f"Excluded ID/time columns:   {len(EXCLUDED_FOR_CLUSTERING)}")
print(f"Numeric ML features:        {len(numeric_features)}")
print(f"Other excluded columns:     {non_numeric_features}")

if len(numeric_features) == 0:
    raise ValueError(
        "STOP — no numeric ML features available."
    )

# ------------------------------------------------------------
# 6. DOCUMENT MISSINGNESS
# ------------------------------------------------------------

IP_REPUTATION_COLUMNS = [
    "ip_reputation_mean",
    "ip_reputation_min",
    "ip_reputation_max",
    "ip_reputation_range",
    "ip_reputation_abs_mean",
]

missing_before = {
    c: int(df[c].isna().sum())
    for c in numeric_features
    if int(df[c].isna().sum()) > 0
}

print("\n[6] MISSING VALUES BEFORE PREPROCESSING")

if missing_before:
    for c, n in missing_before.items():
        print(f"{c}: {n}")
else:
    print("None")

# Verify documented IP reputation state.
for c in IP_REPUTATION_COLUMNS:
    if c in df.columns:
        count_missing = int(df[c].isna().sum())
        if count_missing != 11880:
            raise ValueError(
                f"STOP — unexpected missingness in {c}: "
                f"{count_missing} instead of 11880"
            )

print("Documented IP reputation missingness: PASS")

# ------------------------------------------------------------
# 7. CREATE ML MATRIX
# ------------------------------------------------------------

X = df[numeric_features].copy()

# We DO NOT invent IP reputation values.
# Instead, remove features that are unavailable for every session.
#
# This prevents all-NaN columns from breaking StandardScaler
# while preserving the original v4 dataset untouched.

usable_features = [
    c for c in numeric_features
    if not X[c].isna().all()
]

dropped_all_missing = [
    c for c in numeric_features
    if X[c].isna().all()
]

print("\n[7] ML FEATURE MATRIX")
print(f"Numeric features before filtering: {len(numeric_features)}")
print(f"Usable numeric features:            {len(usable_features)}")
print(f"All-missing features excluded:      {len(dropped_all_missing)}")

if dropped_all_missing:
    print("Excluded all-missing features:")
    for c in dropped_all_missing:
        print(f"  - {c}")

X = X[usable_features].copy()

# ------------------------------------------------------------
# 8. HANDLE REMAINING MISSING VALUES
# ------------------------------------------------------------

remaining_missing = {
    c: int(X[c].isna().sum())
    for c in X.columns
    if int(X[c].isna().sum()) > 0
}

print("\n[8] REMAINING MISSING VALUES")

if remaining_missing:
    for c, n in remaining_missing.items():
        print(f"{c}: {n}")
else:
    print("None")

# For Geo-derived features, missingness was already documented
# as resolved in v4. This is a defensive check.
#
# If any unexpected missing value remains, STOP rather than
# silently imputing it.

if remaining_missing:
    raise ValueError(
        "STOP — unexpected missing values remain in ML features.\n"
        f"Columns: {remaining_missing}\n"
        "No automatic imputation performed."
    )

print("Missing-value handling: PASS")

# ------------------------------------------------------------
# 9. FINITE CHECK BEFORE SCALING
# ------------------------------------------------------------

nonfinite = {}

for c in X.columns:
    values = pd.to_numeric(X[c], errors="coerce").to_numpy()
    bad = int(np.sum(~np.isfinite(values)))
    if bad:
        nonfinite[c] = bad

print("\n[9] FINITE CHECK")

if nonfinite:
    print(nonfinite)
    raise ValueError(
        "STOP — non-finite values remain before scaling."
    )

print("All ML values finite: PASS")

# ------------------------------------------------------------
# 10. STANDARDIZATION
# ------------------------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=usable_features,
    index=df.index
)

print("\n[10] STANDARD SCALING")
print(f"Input shape:  {X.shape}")
print(f"Output shape: {X_scaled.shape}")

# ------------------------------------------------------------
# 11. POST-SCALING SANITY
# ------------------------------------------------------------

post_nonfinite = {}

for c in X_scaled.columns:
    values = X_scaled[c].to_numpy()
    bad = int(np.sum(~np.isfinite(values)))
    if bad:
        post_nonfinite[c] = bad

if post_nonfinite:
    raise ValueError(
        f"STOP — non-finite values after scaling: {post_nonfinite}"
    )

print("Scaled matrix finite: PASS")

# ------------------------------------------------------------
# 12. BUILD ML-READY DATASET
# ------------------------------------------------------------

ml_ready = pd.concat(
    [
        df[[SESSION_ID]].copy(),
        X_scaled
    ],
    axis=1
)

if len(ml_ready) != 11880:
    raise ValueError(
        "STOP — row count changed during preprocessing."
    )

if ml_ready[SESSION_ID].duplicated().any():
    raise ValueError(
        "STOP — duplicate session IDs introduced."
    )

# ------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------

ml_ready.to_csv(OUTPUT_CSV, index=False)

output_sha = sha256_file(OUTPUT_CSV)

# Save scaler parameters as JSON for reproducibility.
scaler_params = {
    "method": "StandardScaler",
    "feature_count": len(usable_features),
    "features": usable_features,
    "mean": {
        c: float(v)
        for c, v in zip(usable_features, scaler.mean_)
    },
    "scale": {
        c: float(v)
        for c, v in zip(usable_features, scaler.scale_)
    },
    "variance": {
        c: float(v)
        for c, v in zip(usable_features, scaler.var_)
    },
    "dropped_all_missing_features": dropped_all_missing,
}

with open(SCALER_PATH, "w", encoding="utf-8") as f:
    json.dump(scaler_params, f, indent=2)

# SHA manifest
with open(SHA_PATH, "w", encoding="utf-8") as f:
    f.write(f"{output_sha}  {os.path.basename(OUTPUT_CSV)}\n")

# ------------------------------------------------------------
# 14. AUDIT
# ------------------------------------------------------------

audit = {
    "step": "6B",
    "title": "Unsupervised ML Preprocessing",
    "source": SOURCE,
    "source_sha256": actual_source_sha,
    "output": OUTPUT_CSV,
    "output_sha256": output_sha,
    "rows": int(len(ml_ready)),
    "source_columns": int(len(df.columns)),
    "numeric_features_before_filter": int(len(numeric_features)),
    "usable_numeric_features": int(len(usable_features)),
    "dropped_all_missing_features": dropped_all_missing,
    "excluded_id_time_columns": sorted(EXCLUDED_FOR_CLUSTERING),
    "remaining_missing_values": remaining_missing,
    "standard_scaling": True,
    "scaler_parameters": SCALER_PATH,
    "label_leakage": False,
    "source_modified": False,
    "sessions_preserved": True,
    "duplicate_sessions": int(ml_ready[SESSION_ID].duplicated().sum()),
    "finite_after_scaling": True,
}

with open(AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2)

audit_sha = sha256_file(AUDIT_PATH)

# ------------------------------------------------------------
# 15. SOURCE PROTECTION
# ------------------------------------------------------------

source_sha_after = sha256_file(SOURCE)

print("\n[11] SOURCE PROTECTION")
print(f"SHA before: {actual_source_sha}")
print(f"SHA after:  {source_sha_after}")

if source_sha_after != SOURCE_SHA:
    raise ValueError(
        "STOP — source dataset changed!"
    )

print("Source unchanged: PASS")

# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 6B — COMPLETE")
print("=" * 70)

print(f"Input:")
print(SOURCE)

print(f"\nInput SHA:")
print(SOURCE_SHA)

print(f"\nOutput:")
print(OUTPUT_CSV)

print(f"Output shape:")
print(ml_ready.shape)

print(f"\nOutput SHA-256:")
print(output_sha)

print(f"\nScaler parameters:")
print(SCALER_PATH)

print(f"\nAudit:")
print(AUDIT_PATH)

print(f"Audit SHA-256:")
print(audit_sha)

print(f"\nSHA manifest:")
print(SHA_PATH)

print("\nSafety:")
print("Source modified:              NO")
print("Sessions preserved:           YES")
print("Session IDs preserved:        YES")
print("Labels used:                  NO")
print("Label leakage:               NO")
print("Artificial IP reputation:    NO")
print("Unexpected missing values:   NO")
print("Scaled matrix finite:         YES")

print("\nSTATUS: PASS")
print("NEXT: STEP 6C — K-MEANS CLUSTERING")

STEP 6B — UNSUPERVISED ML PREPROCESSING

[1] SOURCE
Path: /content/session_behavioral_features_v4/session_behavioral_features_v4.csv
Exists: PASS
Expected SHA: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual SHA:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Source SHA verification: PASS

[2] DATASET
Shape: (11880, 56)
Rows: 11,880
Columns: 56

[3] SESSION INTEGRITY
Missing session IDs:   0
Duplicate session IDs: 0
Unique session IDs:    11880
Session integrity: PASS

[4] LABEL LEAKAGE
Forbidden columns present: []
Label leakage: NO

[5] FEATURE SELECTION
Total source columns:       56
Excluded ID/time columns:   3
Numeric ML features:        53
Other excluded columns:     []

[6] MISSING VALUES BEFORE PREPROCESSING
ip_reputation_mean: 11880
ip_reputation_min: 11880
ip_reputation_max: 11880
ip_reputation_range: 11880
ip_reputation_abs_mean: 11880
Documented IP reputation missingness: PASS

[7] ML FEATURE MATRIX
Numeric features before filter

In [ ]:
# ============================================================
# STEP 6C — K-MEANS CLUSTERING
# ============================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

SOURCE = (
    "/content/step6_unsupervised_ml/step6b_preprocessing/"
    "session_behavioral_features_v4_ml_ready.csv"
)

EXPECTED_SHA = (
    "bf38b32554f91ff24b334415930656cee7d7b1e3ecf93dd584fbfcb059a050a4"
)

OUTPUT_DIR = "/content/step6_unsupervised_ml/step6c_kmeans"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS_PATH = os.path.join(
    OUTPUT_DIR,
    "kmeans_model_comparison.csv"
)

ASSIGNMENTS_PATH = os.path.join(
    OUTPUT_DIR,
    "kmeans_cluster_assignments.csv"
)

AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "step6c_kmeans_audit.json"
)

SHA_PATH = os.path.join(
    OUTPUT_DIR,
    "kmeans_cluster_assignments.sha256"
)

# Candidate k values.
K_VALUES = range(2, 11)

RANDOM_STATE = 42
N_INIT = 20
MAX_ITER = 300

# ------------------------------------------------------------
# SHA256
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


print("=" * 70)
print("STEP 6C — K-MEANS CLUSTERING")
print("=" * 70)

# ------------------------------------------------------------
# 1. SOURCE VERIFICATION
# ------------------------------------------------------------

if not os.path.exists(SOURCE):
    raise FileNotFoundError(
        f"STOP — ML-ready dataset not found:\n{SOURCE}"
    )

actual_sha = sha256_file(SOURCE)

print("\n[1] SOURCE")
print(f"Path: {SOURCE}")
print(f"Exists: PASS")
print(f"Expected SHA: {EXPECTED_SHA}")
print(f"Actual SHA:   {actual_sha}")

if actual_sha != EXPECTED_SHA:
    raise ValueError(
        "STOP — ML-ready dataset SHA mismatch."
    )

print("SHA verification: PASS")

# ------------------------------------------------------------
# 2. LOAD
# ------------------------------------------------------------

df = pd.read_csv(SOURCE)

print("\n[2] DATASET")
print(f"Shape: {df.shape}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

if len(df) != 11880:
    raise ValueError(
        f"STOP — expected 11,880 sessions, got {len(df)}"
    )

# ------------------------------------------------------------
# 3. SESSION ID VALIDATION
# ------------------------------------------------------------

SESSION_ID = "reconstructed_session_id"

if SESSION_ID not in df.columns:
    raise ValueError(
        "STOP — reconstructed_session_id missing."
    )

missing_ids = int(df[SESSION_ID].isna().sum())
duplicate_ids = int(df[SESSION_ID].duplicated().sum())
unique_ids = int(df[SESSION_ID].nunique())

print("\n[3] SESSION INTEGRITY")
print(f"Missing IDs:   {missing_ids}")
print(f"Duplicate IDs: {duplicate_ids}")
print(f"Unique IDs:    {unique_ids}")

if missing_ids != 0:
    raise ValueError("STOP — missing session IDs.")

if duplicate_ids != 0:
    raise ValueError("STOP — duplicate session IDs.")

if unique_ids != 11880:
    raise ValueError("STOP — unexpected session count.")

print("Session integrity: PASS")

# ------------------------------------------------------------
# 4. LABEL LEAKAGE CHECK
# ------------------------------------------------------------

FORBIDDEN_LABELS = {
    "attack_type",
    "attack_stage",
    "severity",
    "mitre_technique",
}

present_forbidden = sorted(
    c for c in df.columns
    if c in FORBIDDEN_LABELS
)

print("\n[4] LABEL LEAKAGE")
print(f"Forbidden columns present: {present_forbidden}")

if present_forbidden:
    raise ValueError(
        "STOP — label columns detected in ML input."
    )

print("Label leakage: NO")

# ------------------------------------------------------------
# 5. FEATURE / ID SEPARATION
# ------------------------------------------------------------

feature_columns = [
    c for c in df.columns
    if c != SESSION_ID
]

X = df[feature_columns].copy()

# Everything should be numeric at this point.
non_numeric = [
    c for c in X.columns
    if not pd.api.types.is_numeric_dtype(X[c])
]

print("\n[5] ML FEATURE MATRIX")
print(f"Feature count: {len(feature_columns)}")
print(f"Non-numeric columns: {non_numeric}")

if non_numeric:
    raise ValueError(
        "STOP — non-numeric columns detected in ML matrix."
    )

# ------------------------------------------------------------
# 6. FINITE CHECK
# ------------------------------------------------------------

X_values = X.to_numpy(dtype=float)

finite_mask = np.isfinite(X_values)

bad_count = int((~finite_mask).sum())

print("\n[6] NUMERIC SANITY")
print(f"Non-finite values: {bad_count}")

if bad_count != 0:
    raise ValueError(
        "STOP — non-finite values detected."
    )

print("Finite numeric matrix: PASS")

# ------------------------------------------------------------
# 7. K-MEANS MODEL COMPARISON
# ------------------------------------------------------------

print("\n[7] K-MEANS MODEL SEARCH")
print(
    f"Testing k values: {list(K_VALUES)}"
)

results = []
models = {}

for k in K_VALUES:

    print(f"Running k={k} ...")

    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=N_INIT,
        max_iter=MAX_ITER
    )

    labels = model.fit_predict(X_values)

    unique_clusters = len(np.unique(labels))

    if unique_clusters < 2:
        raise ValueError(
            f"STOP — k={k} produced fewer than 2 clusters."
        )

    silhouette = silhouette_score(
        X_values,
        labels
    )

    davies_bouldin = davies_bouldin_score(
        X_values,
        labels
    )

    calinski_harabasz = calinski_harabasz_score(
        X_values,
        labels
    )

    inertia = float(model.inertia_)

    results.append({
        "k": k,
        "inertia": inertia,
        "silhouette_score": float(silhouette),
        "davies_bouldin_score": float(davies_bouldin),
        "calinski_harabasz_score": float(
            calinski_harabasz
        ),
        "clusters_found": unique_clusters
    })

    models[k] = model

results_df = pd.DataFrame(results)

# ------------------------------------------------------------
# 8. SELECT BEST K
# ------------------------------------------------------------

# Primary criterion:
# highest Silhouette Score.
#
# Davies-Bouldin is secondary:
# lower is better.
#
# Calinski-Harabasz is additional evidence:
# higher is better.

results_df = results_df.sort_values(
    by=[
        "silhouette_score",
        "davies_bouldin_score",
        "calinski_harabasz_score"
    ],
    ascending=[
        False,
        True,
        False
    ]
).reset_index(drop=True)

best_k = int(results_df.iloc[0]["k"])

best_model = models[best_k]

best_labels = best_model.labels_

print("\n[8] MODEL SELECTION")
print(f"Best k by primary Silhouette criterion: {best_k}")

print("\nModel comparison:")
print(
    results_df.to_string(index=False)
)

# ------------------------------------------------------------
# 9. FINAL CLUSTER ASSIGNMENTS
# ------------------------------------------------------------

assignments = pd.DataFrame({
    SESSION_ID: df[SESSION_ID].values,
    "kmeans_cluster": best_labels.astype(int)
})

# Cluster numbering is algorithmic only.
# It does NOT represent attack severity or class labels.

cluster_counts = (
    assignments["kmeans_cluster"]
    .value_counts()
    .sort_index()
)

print("\n[9] FINAL CLUSTERS")
print(f"Selected k: {best_k}")

print("\nCluster sizes:")

for cluster_id, count in cluster_counts.items():
    percentage = (count / len(assignments)) * 100

    print(
        f"Cluster {cluster_id}: "
        f"{count:,} sessions "
        f"({percentage:.2f}%)"
    )

# ------------------------------------------------------------
# 10. ASSIGNMENT INTEGRITY
# ------------------------------------------------------------

if len(assignments) != 11880:
    raise ValueError(
        "STOP — cluster assignment row count changed."
    )

if assignments[SESSION_ID].duplicated().any():
    raise ValueError(
        "STOP — duplicate session IDs in assignments."
    )

if assignments["kmeans_cluster"].isna().any():
    raise ValueError(
        "STOP — missing cluster assignments."
    )

print("\n[10] ASSIGNMENT INTEGRITY")
print("Rows preserved: YES")
print("Session IDs preserved: YES")
print("Cluster labels complete: YES")

# ------------------------------------------------------------
# 11. SAVE RESULTS
# ------------------------------------------------------------

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

assignments.to_csv(
    ASSIGNMENTS_PATH,
    index=False
)

assignments_sha = sha256_file(
    ASSIGNMENTS_PATH
)

with open(SHA_PATH, "w", encoding="utf-8") as f:
    f.write(
        f"{assignments_sha}  "
        f"{os.path.basename(ASSIGNMENTS_PATH)}\n"
    )

# ------------------------------------------------------------
# 12. AUDIT
# ------------------------------------------------------------

audit = {
    "step": "6C",
    "title": "K-Means Clustering",
    "source": SOURCE,
    "source_sha256": actual_sha,
    "rows": int(len(df)),
    "features_used": int(len(feature_columns)),
    "random_state": RANDOM_STATE,
    "n_init": N_INIT,
    "max_iter": MAX_ITER,
    "tested_k_values": list(K_VALUES),
    "selected_k": best_k,
    "selection_primary_metric": "silhouette_score",
    "selection_secondary_metric": "davies_bouldin_score",
    "label_leakage": False,
    "session_count_preserved": True,
    "duplicate_session_ids": duplicate_ids,
    "cluster_sizes": {
        str(int(k)): int(v)
        for k, v in cluster_counts.items()
    },
    "results_path": RESULTS_PATH,
    "assignments_path": ASSIGNMENTS_PATH,
    "assignments_sha256": assignments_sha,
    "source_modified": False
}

with open(AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2)

audit_sha = sha256_file(AUDIT_PATH)

# ------------------------------------------------------------
# 13. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 6C — COMPLETE")
print("=" * 70)

print(f"\nInput:")
print(SOURCE)

print("\nInput SHA:")
print(actual_sha)

print("\nModel comparison:")
print(RESULTS_PATH)

print("\nCluster assignments:")
print(ASSIGNMENTS_PATH)

print("\nAssignments SHA-256:")
print(assignments_sha)

print("\nSHA manifest:")
print(SHA_PATH)

print("\nAudit:")
print(AUDIT_PATH)

print("\nAudit SHA-256:")
print(audit_sha)

print("\nSelected k:")
print(best_k)

print("\nSafety:")
print("Source modified:          NO")
print("Sessions preserved:       YES")
print("Session IDs preserved:    YES")
print("Labels used:              NO")
print("Label leakage:            NO")
print("Non-finite values:        NO")
print("Cluster assignments:      COMPLETE")

print("\nSTATUS: PASS")
print("NEXT: STEP 6D — DBSCAN")

STEP 6C — K-MEANS CLUSTERING

[1] SOURCE
Path: /content/step6_unsupervised_ml/step6b_preprocessing/session_behavioral_features_v4_ml_ready.csv
Exists: PASS
Expected SHA: bf38b32554f91ff24b334415930656cee7d7b1e3ecf93dd584fbfcb059a050a4
Actual SHA:   bf38b32554f91ff24b334415930656cee7d7b1e3ecf93dd584fbfcb059a050a4
SHA verification: PASS

[2] DATASET
Shape: (11880, 49)
Rows: 11,880
Columns: 49

[3] SESSION INTEGRITY
Missing IDs:   0
Duplicate IDs: 0
Unique IDs:    11880
Session integrity: PASS

[4] LABEL LEAKAGE
Forbidden columns present: []
Label leakage: NO

[5] ML FEATURE MATRIX
Feature count: 48
Non-numeric columns: []

[6] NUMERIC SANITY
Non-finite values: 0
Finite numeric matrix: PASS

[7] K-MEANS MODEL SEARCH
Testing k values: [2, 3, 4, 5, 6, 7, 8, 9, 10]
Running k=2 ...
Running k=3 ...
Running k=4 ...
Running k=5 ...
Running k=6 ...
Running k=7 ...
Running k=8 ...
Running k=9 ...
Running k=10 ...

[8] MODEL SELECTION
Best k by primary Silhouette criterion: 5

Model comparison:
 k 

In [ ]:
# ============================================================
# STEP 6D — DBSCAN CLUSTERING
# ============================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd

from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

INPUT = "/content/step6_unsupervised_ml/step6b_preprocessing/session_behavioral_features_v4_ml_ready.csv"
INPUT_SHA = "/content/step6_unsupervised_ml/step6b_preprocessing/session_behavioral_features_v4_ml_ready.sha256"

OUT_DIR = "/content/step6_unsupervised_ml/step6d_dbscan"
os.makedirs(OUT_DIR, exist_ok=True)

COMPARISON_OUT = os.path.join(
    OUT_DIR,
    "dbscan_model_comparison.csv"
)

ASSIGNMENTS_OUT = os.path.join(
    OUT_DIR,
    "dbscan_cluster_assignments.csv"
)

ASSIGNMENTS_SHA_OUT = os.path.join(
    OUT_DIR,
    "dbscan_cluster_assignments.sha256"
)

AUDIT_OUT = os.path.join(
    OUT_DIR,
    "step6d_dbscan_audit.json"
)

AUDIT_SHA_OUT = os.path.join(
    OUT_DIR,
    "step6d_dbscan_audit.sha256"
)


# ------------------------------------------------------------
# SHA-256 FUNCTION
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# [1] SOURCE VERIFICATION
# ------------------------------------------------------------

print("=" * 70)
print("STEP 6D — DBSCAN CLUSTERING")
print("=" * 70)

if not os.path.exists(INPUT):
    raise FileNotFoundError(
        f"STOP — ML-ready input not found:\n{INPUT}"
    )

print("\n[1] SOURCE")
print(f"Path: {INPUT}")
print("Exists: PASS")

EXPECTED_SHA = None

if os.path.exists(INPUT_SHA):
    with open(INPUT_SHA, "r", encoding="utf-8") as f:
        manifest_text = f.read().strip()

    if manifest_text:
        EXPECTED_SHA = manifest_text.split()[0]

ACTUAL_SHA = sha256_file(INPUT)

print(f"Expected SHA: {EXPECTED_SHA}")
print(f"Actual SHA:   {ACTUAL_SHA}")

if EXPECTED_SHA is None:
    raise ValueError(
        "STOP — SHA manifest exists but expected SHA could not be read."
    )

if ACTUAL_SHA != EXPECTED_SHA:
    raise ValueError(
        "\nSTOP — SHA-256 MISMATCH.\n"
        "The DBSCAN input does not match the verified ML-ready checkpoint."
    )

print("SHA verification: PASS")


# ------------------------------------------------------------
# [2] LOAD DATASET
# ------------------------------------------------------------

df = pd.read_csv(INPUT)

print("\n[2] DATASET")
print(f"Shape: {df.shape}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

if len(df) != 11880:
    raise ValueError(
        f"STOP — expected 11,880 sessions, found {len(df)}."
    )

print("Row count: PASS")


# ------------------------------------------------------------
# [3] SESSION INTEGRITY
# ------------------------------------------------------------

SESSION_ID = "reconstructed_session_id"

if SESSION_ID not in df.columns:
    raise ValueError(
        f"STOP — missing session ID column: {SESSION_ID}"
    )

missing_ids = int(df[SESSION_ID].isna().sum())
duplicate_ids = int(df[SESSION_ID].duplicated().sum())
unique_ids = int(df[SESSION_ID].nunique())

print("\n[3] SESSION INTEGRITY")
print(f"Missing IDs:   {missing_ids}")
print(f"Duplicate IDs: {duplicate_ids}")
print(f"Unique IDs:    {unique_ids}")

if missing_ids != 0:
    raise ValueError("STOP — missing session IDs detected.")

if duplicate_ids != 0:
    raise ValueError("STOP — duplicate session IDs detected.")

if unique_ids != 11880:
    raise ValueError(
        f"STOP — expected 11,880 unique sessions, found {unique_ids}."
    )

print("Session integrity: PASS")


# ------------------------------------------------------------
# [4] LABEL LEAKAGE CHECK
# ------------------------------------------------------------

FORBIDDEN = {
    "attack_type",
    "attack_stage",
    "severity",
    "mitre_technique",
    "label",
    "target",
    "y"
}

forbidden_present = [
    c for c in df.columns
    if c.lower() in FORBIDDEN
]

print("\n[4] LABEL LEAKAGE")
print(f"Forbidden label columns present: {forbidden_present}")

if forbidden_present:
    raise ValueError(
        f"STOP — forbidden label columns detected: {forbidden_present}"
    )

print("Label leakage: NO")


# ------------------------------------------------------------
# [5] FEATURE MATRIX
# ------------------------------------------------------------

excluded = {
    SESSION_ID,
    "session_start_time",
    "session_end_time"
}

feature_columns = [
    c for c in df.columns
    if c not in excluded
]

X_df = df[feature_columns].copy()

non_numeric = [
    c for c in X_df.columns
    if not pd.api.types.is_numeric_dtype(X_df[c])
]

print("\n[5] ML FEATURE MATRIX")
print(f"Feature count: {len(feature_columns)}")
print(f"Non-numeric columns: {non_numeric}")

if non_numeric:
    raise ValueError(
        f"STOP — non-numeric ML features detected: {non_numeric}"
    )

X = X_df.to_numpy(dtype=float)

print(f"Matrix shape: {X.shape}")


# ------------------------------------------------------------
# [6] NUMERIC SANITY
# ------------------------------------------------------------

finite_mask = np.isfinite(X)

nonfinite_count = int((~finite_mask).sum())

print("\n[6] NUMERIC SANITY")
print(f"Non-finite values: {nonfinite_count}")

if nonfinite_count != 0:
    raise ValueError(
        "STOP — non-finite values detected in DBSCAN input."
    )

print("Finite numeric matrix: PASS")


# ------------------------------------------------------------
# [7] DBSCAN MODEL SEARCH
# ------------------------------------------------------------

# Conservative grid around the standardized feature space.
# The same X is used for every configuration.

EPS_VALUES = [
    0.50,
    0.75,
    1.00,
    1.25,
    1.50,
    2.00,
    2.50,
    3.00
]

MIN_SAMPLES_VALUES = [
    5,
    10,
    20
]

print("\n[7] DBSCAN MODEL SEARCH")
print(f"eps values: {EPS_VALUES}")
print(f"min_samples values: {MIN_SAMPLES_VALUES}")

results = []
models = {}

for min_samples in MIN_SAMPLES_VALUES:

    for eps in EPS_VALUES:

        print(
            f"Running eps={eps}, "
            f"min_samples={min_samples} ..."
        )

        model = DBSCAN(
            eps=eps,
            min_samples=min_samples,
            metric="euclidean",
            n_jobs=-1
        )

        labels = model.fit_predict(X)

        unique_labels = sorted(set(labels))

        non_noise_labels = [
            x for x in unique_labels
            if x != -1
        ]

        clusters_found = len(non_noise_labels)

        noise_count = int(np.sum(labels == -1))
        noise_fraction = noise_count / len(labels)

        silhouette = np.nan
        davies = np.nan
        calinski = np.nan

        # ----------------------------------------------------
        # Metrics are calculated on NON-NOISE points only.
        # This avoids treating DBSCAN's noise label (-1)
        # as an ordinary cluster.
        # ----------------------------------------------------

        non_noise_mask = labels != -1

        X_eval = X[non_noise_mask]
        labels_eval = labels[non_noise_mask]

        if (
            clusters_found >= 2
            and len(X_eval) > clusters_found
            and len(np.unique(labels_eval)) >= 2
        ):
            try:
                silhouette = float(
                    silhouette_score(
                        X_eval,
                        labels_eval
                    )
                )
            except Exception:
                silhouette = np.nan

            try:
                davies = float(
                    davies_bouldin_score(
                        X_eval,
                        labels_eval
                    )
                )
            except Exception:
                davies = np.nan

            try:
                calinski = float(
                    calinski_harabasz_score(
                        X_eval,
                        labels_eval
                    )
                )
            except Exception:
                calinski = np.nan

        results.append({
            "eps": eps,
            "min_samples": min_samples,
            "clusters_found": clusters_found,
            "noise_count": noise_count,
            "noise_fraction": noise_fraction,
            "silhouette_score": silhouette,
            "davies_bouldin_score": davies,
            "calinski_harabasz_score": calinski
        })

        models[
            (eps, min_samples)
        ] = model


comparison = pd.DataFrame(results)


# ------------------------------------------------------------
# [8] MODEL SELECTION
# ------------------------------------------------------------

print("\n[8] MODEL COMPARISON")

print(
    comparison.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Selection policy:
#
# 1. Need at least 2 real clusters.
# 2. Prefer a meaningful noise fraction, but avoid
#    pathological "everything is noise".
# 3. Primary criterion = highest silhouette.
#
# Noise fraction is NOT used as the only criterion.
# ------------------------------------------------------------

eligible = comparison[
    (comparison["clusters_found"] >= 2) &
    (comparison["noise_fraction"] < 0.95) &
    (comparison["silhouette_score"].notna())
].copy()

if eligible.empty:
    raise ValueError(
        "STOP — no DBSCAN configuration produced "
        "at least 2 valid clusters with a calculable silhouette."
    )

eligible = eligible.sort_values(
    by=[
        "silhouette_score",
        "davies_bouldin_score"
    ],
    ascending=[
        False,
        True
    ]
)

best_row = eligible.iloc[0]

BEST_EPS = float(best_row["eps"])
BEST_MIN_SAMPLES = int(best_row["min_samples"])

print("\n[9] SELECTED MODEL")
print(f"Best eps: {BEST_EPS}")
print(f"Best min_samples: {BEST_MIN_SAMPLES}")
print(
    f"Clusters found: "
    f"{int(best_row['clusters_found'])}"
)
print(
    f"Noise count: "
    f"{int(best_row['noise_count']):,}"
)
print(
    f"Noise fraction: "
    f"{best_row['noise_fraction']:.4f}"
)
print(
    f"Silhouette: "
    f"{best_row['silhouette_score']:.6f}"
)


# ------------------------------------------------------------
# [10] FINAL DBSCAN ASSIGNMENTS
# ------------------------------------------------------------

final_model = models[
    (BEST_EPS, BEST_MIN_SAMPLES)
]

final_labels = final_model.labels_

assignments = pd.DataFrame({
    SESSION_ID: df[SESSION_ID].astype(str),
    "dbscan_cluster": final_labels
})

if len(assignments) != len(df):
    raise ValueError(
        "STOP — assignment row count changed."
    )

if assignments[SESSION_ID].duplicated().any():
    raise ValueError(
        "STOP — duplicate session IDs in assignments."
    )

if assignments["dbscan_cluster"].isna().any():
    raise ValueError(
        "STOP — incomplete DBSCAN assignments."
    )


# ------------------------------------------------------------
# [11] CLUSTER SUMMARY
# ------------------------------------------------------------

cluster_sizes = (
    assignments["dbscan_cluster"]
    .value_counts()
    .sort_index()
)

print("\n[10] FINAL CLUSTERS")

for cluster_id, count in cluster_sizes.items():

    if cluster_id == -1:
        print(
            f"Noise (-1): {count:,} sessions "
            f"({count / len(assignments) * 100:.2f}%)"
        )
    else:
        print(
            f"Cluster {cluster_id}: {count:,} sessions "
            f"({count / len(assignments) * 100:.2f}%)"
        )


# ------------------------------------------------------------
# [12] SAVE MODEL COMPARISON
# ------------------------------------------------------------

comparison.to_csv(
    COMPARISON_OUT,
    index=False
)


# ------------------------------------------------------------
# [13] SAVE ASSIGNMENTS
# ------------------------------------------------------------

assignments.to_csv(
    ASSIGNMENTS_OUT,
    index=False
)

ASSIGNMENTS_SHA = sha256_file(
    ASSIGNMENTS_OUT
)

with open(
    ASSIGNMENTS_SHA_OUT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        f"{ASSIGNMENTS_SHA}  "
        f"{os.path.basename(ASSIGNMENTS_OUT)}\n"
    )


# ------------------------------------------------------------
# [14] AUDIT
# ------------------------------------------------------------

cluster_count = len(
    [
        x for x in set(final_labels)
        if x != -1
    ]
)

noise_count = int(
    np.sum(final_labels == -1)
)

audit = {
    "step": "6D",
    "name": "DBSCAN clustering",

    "input": INPUT,
    "input_sha256": ACTUAL_SHA,

    "rows": int(len(df)),
    "columns": int(len(df.columns)),

    "session_id_column": SESSION_ID,
    "unique_sessions": unique_ids,

    "feature_count": int(len(feature_columns)),
    "feature_columns": feature_columns,

    "forbidden_label_columns": forbidden_present,
    "label_leakage": False,

    "parameter_search": {
        "eps_values": EPS_VALUES,
        "min_samples_values": MIN_SAMPLES_VALUES
    },

    "selected_model": {
        "eps": BEST_EPS,
        "min_samples": BEST_MIN_SAMPLES,
        "clusters_found": int(cluster_count),
        "noise_count": noise_count,
        "noise_fraction": float(
            noise_count / len(final_labels)
        ),
        "silhouette_score": (
            None
            if pd.isna(best_row["silhouette_score"])
            else float(best_row["silhouette_score"])
        ),
        "davies_bouldin_score": (
            None
            if pd.isna(best_row["davies_bouldin_score"])
            else float(best_row["davies_bouldin_score"])
        ),
        "calinski_harabasz_score": (
            None
            if pd.isna(best_row["calinski_harabasz_score"])
            else float(best_row["calinski_harabasz_score"])
        )
    },

    "cluster_sizes": {
        str(k): int(v)
        for k, v in cluster_sizes.items()
    },

    "assignments": ASSIGNMENTS_OUT,
    "assignments_sha256": ASSIGNMENTS_SHA,

    "model_comparison": COMPARISON_OUT,

    "source_modified": False,
    "sessions_preserved": True,
    "session_ids_preserved": True,
    "labels_used": False,
    "label_leakage": False,
    "nonfinite_values": nonfinite_count,
    "assignments_complete": True
}

with open(
    AUDIT_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False
    )

AUDIT_SHA = sha256_file(
    AUDIT_OUT
)

with open(
    AUDIT_SHA_OUT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        f"{AUDIT_SHA}  "
        f"{os.path.basename(AUDIT_OUT)}\n"
    )


# ------------------------------------------------------------
# [15] FINAL SOURCE PROTECTION CHECK
# ------------------------------------------------------------

FINAL_INPUT_SHA = sha256_file(INPUT)

print("\n[11] SOURCE PROTECTION")
print(f"SHA before: {ACTUAL_SHA}")
print(f"SHA after:  {FINAL_INPUT_SHA}")

if FINAL_INPUT_SHA != ACTUAL_SHA:
    raise ValueError(
        "STOP — source dataset changed during DBSCAN."
    )

print("Source unchanged: PASS")


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 6D — COMPLETE")
print("=" * 70)

print(f"\nInput:")
print(INPUT)

print(f"\nInput SHA:")
print(ACTUAL_SHA)

print("\nModel comparison:")
print(COMPARISON_OUT)

print("\nCluster assignments:")
print(ASSIGNMENTS_OUT)

print("\nAssignments SHA-256:")
print(ASSIGNMENTS_SHA)

print("\nSHA manifest:")
print(ASSIGNMENTS_SHA_OUT)

print("\nAudit:")
print(AUDIT_OUT)

print("\nAudit SHA-256:")
print(AUDIT_SHA)

print("\nSelected eps:")
print(BEST_EPS)

print("\nSelected min_samples:")
print(BEST_MIN_SAMPLES)

print("\nSelected clusters:")
print(cluster_count)

print("\nNoise sessions:")
print(noise_count)

print("\nSafety:")
print("Source modified:          NO")
print("Sessions preserved:       YES")
print("Session IDs preserved:    YES")
print("Labels used:              NO")
print("Label leakage:            NO")
print("Non-finite values:        NO")
print("Cluster assignments:      COMPLETE")

print("\nSTATUS: PASS")
print("NEXT: STEP 6E — AGGLOMERATIVE CLUSTERING")

STEP 6D — DBSCAN CLUSTERING

[1] SOURCE
Path: /content/step6_unsupervised_ml/step6b_preprocessing/session_behavioral_features_v4_ml_ready.csv
Exists: PASS
Expected SHA: bf38b32554f91ff24b334415930656cee7d7b1e3ecf93dd584fbfcb059a050a4
Actual SHA:   bf38b32554f91ff24b334415930656cee7d7b1e3ecf93dd584fbfcb059a050a4
SHA verification: PASS

[2] DATASET
Shape: (11880, 49)
Rows: 11,880
Columns: 49
Row count: PASS

[3] SESSION INTEGRITY
Missing IDs:   0
Duplicate IDs: 0
Unique IDs:    11880
Session integrity: PASS

[4] LABEL LEAKAGE
Forbidden label columns present: []
Label leakage: NO

[5] ML FEATURE MATRIX
Feature count: 48
Non-numeric columns: []
Matrix shape: (11880, 48)

[6] NUMERIC SANITY
Non-finite values: 0
Finite numeric matrix: PASS

[7] DBSCAN MODEL SEARCH
eps values: [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0]
min_samples values: [5, 10, 20]
Running eps=0.5, min_samples=5 ...
Running eps=0.75, min_samples=5 ...
Running eps=1.0, min_samples=5 ...
Running eps=1.25, min_samples=5 ...
Ru

In [ ]:
# ================================================================
# STEP 6E — AGGLOMERATIVE CLUSTERING
# ================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# ----------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------

INPUT = "/content/step6_unsupervised_ml/step6b_preprocessing/session_behavioral_features_v4_ml_ready.csv"
EXPECTED_SHA = "bf38b32554f91ff24b334415930656cee7d7b1e3ecf93dd584fbfcb059a050a4"

OUT_DIR = "/content/step6_unsupervised_ml/step6e_agglomerative"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_COMPARISON = os.path.join(
    OUT_DIR, "agglomerative_model_comparison.csv"
)

ASSIGNMENTS = os.path.join(
    OUT_DIR, "agglomerative_cluster_assignments.csv"
)

ASSIGNMENTS_SHA = os.path.join(
    OUT_DIR, "agglomerative_cluster_assignments.sha256"
)

AUDIT = os.path.join(
    OUT_DIR, "step6e_agglomerative_audit.json"
)


# ----------------------------------------------------------------
# SHA-256
# ----------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


# ----------------------------------------------------------------
# [1] SOURCE VERIFICATION
# ----------------------------------------------------------------

print("=" * 70)
print("STEP 6E — AGGLOMERATIVE CLUSTERING")
print("=" * 70)

print("\n[1] SOURCE")

if not os.path.exists(INPUT):
    raise FileNotFoundError(f"STOP — source not found: {INPUT}")

actual_sha = sha256_file(INPUT)

print(f"Path: {INPUT}")
print("Exists: PASS")
print(f"Expected SHA: {EXPECTED_SHA}")
print(f"Actual SHA:   {actual_sha}")

if actual_sha != EXPECTED_SHA:
    raise ValueError(
        "STOP — source SHA-256 mismatch."
    )

print("SHA verification: PASS")


# ----------------------------------------------------------------
# [2] LOAD DATA
# ----------------------------------------------------------------

df = pd.read_csv(INPUT)

print("\n[2] DATASET")
print(f"Shape: {df.shape}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

if len(df) != 11880:
    raise ValueError("STOP — unexpected row count.")

print("Row count: PASS")


# ----------------------------------------------------------------
# [3] SESSION INTEGRITY
# ----------------------------------------------------------------

SESSION_ID = "reconstructed_session_id"

if SESSION_ID not in df.columns:
    raise ValueError("STOP — session ID missing.")

missing_ids = int(df[SESSION_ID].isna().sum())
duplicate_ids = int(df[SESSION_ID].duplicated().sum())
unique_ids = int(df[SESSION_ID].nunique())

print("\n[3] SESSION INTEGRITY")
print(f"Missing IDs:   {missing_ids}")
print(f"Duplicate IDs: {duplicate_ids}")
print(f"Unique IDs:    {unique_ids}")

if missing_ids != 0 or duplicate_ids != 0 or unique_ids != len(df):
    raise ValueError(
        "STOP — session integrity failure."
    )

print("Session integrity: PASS")


# ----------------------------------------------------------------
# [4] LABEL LEAKAGE
# ----------------------------------------------------------------

FORBIDDEN = {
    "attack_type",
    "attack_stage",
    "severity",
    "mitre_technique"
}

forbidden_present = sorted(
    FORBIDDEN.intersection(df.columns)
)

print("\n[4] LABEL LEAKAGE")
print(f"Forbidden label columns present: {forbidden_present}")

if forbidden_present:
    raise ValueError(
        f"STOP — forbidden label columns detected: {forbidden_present}"
    )

print("Label leakage: NO")


# ----------------------------------------------------------------
# [5] FEATURE SELECTION
# ----------------------------------------------------------------

excluded_columns = {
    SESSION_ID,
    "session_start_time",
    "session_end_time"
}

feature_columns = [
    c for c in df.columns
    if c not in excluded_columns
]

X_df = df[feature_columns].copy()

non_numeric = X_df.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("\n[5] ML FEATURE MATRIX")
print(f"Feature count: {len(feature_columns)}")
print(f"Non-numeric columns: {non_numeric}")

if non_numeric:
    raise ValueError(
        f"STOP — non-numeric ML features detected: {non_numeric}"
    )

X = X_df.to_numpy(dtype=float)

print(f"Matrix shape: {X.shape}")


# ----------------------------------------------------------------
# [6] NUMERIC SANITY
# ----------------------------------------------------------------

nonfinite_count = int(
    np.sum(~np.isfinite(X))
)

print("\n[6] NUMERIC SANITY")
print(f"Non-finite values: {nonfinite_count}")

if nonfinite_count != 0:
    raise ValueError(
        "STOP — non-finite values detected."
    )

print("Finite numeric matrix: PASS")


# ----------------------------------------------------------------
# [7] AGGLOMERATIVE MODEL SEARCH
# ----------------------------------------------------------------

# We use the same candidate k range used for K-Means.
K_VALUES = [2, 3, 4, 5, 6, 7, 8, 9, 10]

print("\n[7] AGGLOMERATIVE MODEL SEARCH")
print(f"Testing cluster counts: {K_VALUES}")

results = []
models = {}

for k in K_VALUES:

    print(f"Running k={k} ...")

    model = AgglomerativeClustering(
        n_clusters=k,
        linkage="ward"
    )

    labels = model.fit_predict(X)

    clusters_found = len(np.unique(labels))

    if clusters_found < 2:
        silhouette = np.nan
        davies = np.nan
        calinski = np.nan
    else:
        silhouette = silhouette_score(X, labels)
        davies = davies_bouldin_score(X, labels)
        calinski = calinski_harabasz_score(X, labels)

    inertia_like = np.nan

    # Ward linkage provides a hierarchical criterion,
    # but sklearn's AgglomerativeClustering does not expose
    # the same K-Means inertia. Therefore we do NOT invent one.

    results.append({
        "k": k,
        "silhouette_score": silhouette,
        "davies_bouldin_score": davies,
        "calinski_harabasz_score": calinski,
        "clusters_found": clusters_found
    })

    models[k] = model


comparison = pd.DataFrame(results)

# Primary criterion:
# highest silhouette
best_row = comparison.loc[
    comparison["silhouette_score"].idxmax()
]

best_k = int(best_row["k"])

print("\n[8] MODEL SELECTION")
print(f"Best k by primary Silhouette criterion: {best_k}")

print("\nModel comparison:")
print(
    comparison
    .sort_values("silhouette_score", ascending=False)
    .to_string(index=False)
)


# ----------------------------------------------------------------
# [9] FINAL MODEL
# ----------------------------------------------------------------

final_model = AgglomerativeClustering(
    n_clusters=best_k,
    linkage="ward"
)

final_labels = final_model.fit_predict(X)

df_assignments = pd.DataFrame({
    SESSION_ID: df[SESSION_ID].values,
    "agglomerative_cluster": final_labels
})

cluster_sizes = (
    df_assignments["agglomerative_cluster"]
    .value_counts()
    .sort_index()
)

print("\n[9] FINAL CLUSTERS")
print(f"Selected k: {best_k}")

for cluster_id, count in cluster_sizes.items():
    pct = count / len(df_assignments) * 100
    print(
        f"Cluster {cluster_id}: "
        f"{count:,} sessions ({pct:.2f}%)"
    )


# ----------------------------------------------------------------
# [10] ASSIGNMENT INTEGRITY
# ----------------------------------------------------------------

print("\n[10] ASSIGNMENT INTEGRITY")

assignment_missing = int(
    df_assignments["agglomerative_cluster"].isna().sum()
)

assignment_rows = len(df_assignments)
assignment_unique_ids = df_assignments[SESSION_ID].nunique()

print(f"Rows preserved: {assignment_rows == len(df)}")
print(f"Session IDs preserved: {assignment_unique_ids == len(df)}")
print(f"Missing cluster labels: {assignment_missing}")

if (
    assignment_rows != len(df)
    or assignment_unique_ids != len(df)
    or assignment_missing != 0
):
    raise ValueError(
        "STOP — cluster assignment integrity failure."
    )

print("Cluster assignments: COMPLETE")


# ----------------------------------------------------------------
# [11] SAVE OUTPUTS
# ----------------------------------------------------------------

comparison.to_csv(
    MODEL_COMPARISON,
    index=False
)

df_assignments.to_csv(
    ASSIGNMENTS,
    index=False
)

assignments_sha = sha256_file(ASSIGNMENTS)

with open(ASSIGNMENTS_SHA, "w") as f:
    f.write(f"{assignments_sha}  {os.path.basename(ASSIGNMENTS)}\n")


# ----------------------------------------------------------------
# [12] AUDIT
# ----------------------------------------------------------------

audit = {
    "step": "6E",
    "model": "AgglomerativeClustering",
    "linkage": "ward",
    "input": INPUT,
    "input_sha256": actual_sha,
    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "feature_count": int(len(feature_columns)),
    "session_id": SESSION_ID,
    "session_count": int(unique_ids),
    "k_values_tested": K_VALUES,
    "selected_k": best_k,
    "selection_criterion": "silhouette_score",
    "label_columns_used": [],
    "label_leakage": False,
    "nonfinite_values": nonfinite_count,
    "source_modified": False,
    "assignments_complete": True,
    "assignment_sha256": assignments_sha,
    "cluster_sizes": {
        str(int(k)): int(v)
        for k, v in cluster_sizes.items()
    }
}

with open(AUDIT, "w") as f:
    json.dump(
        audit,
        f,
        indent=2
    )

audit_sha = sha256_file(AUDIT)


# ----------------------------------------------------------------
# [13] FINAL REPORT
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 6E — COMPLETE")
print("=" * 70)

print(f"\nInput:")
print(INPUT)

print(f"\nInput SHA:")
print(actual_sha)

print(f"\nModel comparison:")
print(MODEL_COMPARISON)

print(f"\nCluster assignments:")
print(ASSIGNMENTS)

print(f"\nAssignments SHA-256:")
print(assignments_sha)

print(f"\nSHA manifest:")
print(ASSIGNMENTS_SHA)

print(f"\nAudit:")
print(AUDIT)

print(f"\nAudit SHA-256:")
print(audit_sha)

print(f"\nSelected k:")
print(best_k)

print("\nSafety:")
print("Source modified:          NO")
print("Sessions preserved:       YES")
print("Session IDs preserved:    YES")
print("Labels used:              NO")
print("Label leakage:            NO")
print("Non-finite values:        NO")
print("Cluster assignments:      COMPLETE")

print("\nSTATUS: PASS")
print("NEXT: STEP 6F — UNSUPERVISED MODEL COMPARISON")

STEP 6E — AGGLOMERATIVE CLUSTERING

[1] SOURCE
Path: /content/step6_unsupervised_ml/step6b_preprocessing/session_behavioral_features_v4_ml_ready.csv
Exists: PASS
Expected SHA: bf38b32554f91ff24b334415930656cee7d7b1e3ecf93dd584fbfcb059a050a4
Actual SHA:   bf38b32554f91ff24b334415930656cee7d7b1e3ecf93dd584fbfcb059a050a4
SHA verification: PASS

[2] DATASET
Shape: (11880, 49)
Rows: 11,880
Columns: 49
Row count: PASS

[3] SESSION INTEGRITY
Missing IDs:   0
Duplicate IDs: 0
Unique IDs:    11880
Session integrity: PASS

[4] LABEL LEAKAGE
Forbidden label columns present: []
Label leakage: NO

[5] ML FEATURE MATRIX
Feature count: 48
Non-numeric columns: []
Matrix shape: (11880, 48)

[6] NUMERIC SANITY
Non-finite values: 0
Finite numeric matrix: PASS

[7] AGGLOMERATIVE MODEL SEARCH
Testing cluster counts: [2, 3, 4, 5, 6, 7, 8, 9, 10]
Running k=2 ...
Running k=3 ...
Running k=4 ...
Running k=5 ...
Running k=6 ...
Running k=7 ...
Running k=8 ...
Running k=9 ...
Running k=10 ...

[8] MODEL SELECTIO

In [ ]:
# ================================================================
# STEP 6F — UNSUPERVISED ML AUDIT + MODEL COMPARISON
# ================================================================

import os
import json
import hashlib
import pandas as pd
import numpy as np
from pathlib import Path

# ----------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------

BASE = Path("/content/step6_unsupervised_ml")

SOURCE = BASE / "step6b_preprocessing" / \
    "session_behavioral_features_v4_ml_ready.csv"

KMEANS_ASSIGNMENTS = BASE / "step6c_kmeans" / \
    "kmeans_cluster_assignments.csv"

KMEANS_COMPARISON = BASE / "step6c_kmeans" / \
    "kmeans_model_comparison.csv"

DBSCAN_ASSIGNMENTS = BASE / "step6d_dbscan" / \
    "dbscan_cluster_assignments.csv"

DBSCAN_COMPARISON = BASE / "step6d_dbscan" / \
    "dbscan_model_comparison.csv"

AGG_ASSIGNMENTS = BASE / "step6e_agglomerative" / \
    "agglomerative_cluster_assignments.csv"

AGG_COMPARISON = BASE / "step6e_agglomerative" / \
    "agglomerative_model_comparison.csv"

OUTDIR = BASE / "step6f_final_audit"
OUTDIR.mkdir(parents=True, exist_ok=True)

FINAL_COMPARISON = OUTDIR / "step6f_model_comparison.csv"
FINAL_AUDIT = OUTDIR / "step6f_unsupervised_ml_audit.json"
FINAL_SHA = OUTDIR / "step6f_unsupervised_ml_audit.sha256"
FINAL_MANIFEST = OUTDIR / "step6f_manifest.json"


# ----------------------------------------------------------------
# SHA-256 HELPER
# ----------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


# ----------------------------------------------------------------
# REQUIRED FILE CHECK
# ----------------------------------------------------------------

required_files = {
    "source": SOURCE,
    "kmeans_assignments": KMEANS_ASSIGNMENTS,
    "kmeans_comparison": KMEANS_COMPARISON,
    "dbscan_assignments": DBSCAN_ASSIGNMENTS,
    "dbscan_comparison": DBSCAN_COMPARISON,
    "agglomerative_assignments": AGG_ASSIGNMENTS,
    "agglomerative_comparison": AGG_COMPARISON,
}

missing = [
    name for name, path in required_files.items()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "STOP — required Step 6 files are missing: "
        + ", ".join(missing)
    )

print("=" * 70)
print("STEP 6F — UNSUPERVISED ML AUDIT + MODEL COMPARISON")
print("=" * 70)


# ----------------------------------------------------------------
# 1. SOURCE SHA
# ----------------------------------------------------------------

EXPECTED_SOURCE_SHA = (
    "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"
)

actual_source_sha = sha256_file(SOURCE)

print("\n[1] SOURCE SHA-256")
print("Expected:", EXPECTED_SOURCE_SHA)
print("Actual:  ", actual_source_sha)

if actual_source_sha != EXPECTED_SOURCE_SHA:
    raise ValueError("STOP — source SHA mismatch.")

print("Verification: PASS")


# ----------------------------------------------------------------
# 2. LOAD DATA
# ----------------------------------------------------------------

source_df = pd.read_csv(SOURCE)

kmeans_assign = pd.read_csv(KMEANS_ASSIGNMENTS)
dbscan_assign = pd.read_csv(DBSCAN_ASSIGNMENTS)
agg_assign = pd.read_csv(AGG_ASSIGNMENTS)

kmeans_comp = pd.read_csv(KMEANS_COMPARISON)
dbscan_comp = pd.read_csv(DBSCAN_COMPARISON)
agg_comp = pd.read_csv(AGG_COMPARISON)

print("\n[2] DATASET")
print("Source shape:", source_df.shape)
print("Rows:", len(source_df))
print("Columns:", len(source_df.columns))


# ----------------------------------------------------------------
# 3. SESSION INTEGRITY
# ----------------------------------------------------------------

SESSION_ID = "reconstructed_session_id"

if SESSION_ID not in source_df.columns:
    raise ValueError("STOP — session ID missing from source.")

missing_ids = int(source_df[SESSION_ID].isna().sum())
duplicate_ids = int(source_df[SESSION_ID].duplicated().sum())
unique_ids = int(source_df[SESSION_ID].nunique())

print("\n[3] SESSION INTEGRITY")
print("Missing IDs:", missing_ids)
print("Duplicate IDs:", duplicate_ids)
print("Unique IDs:", unique_ids)

if missing_ids != 0:
    raise ValueError("STOP — missing session IDs.")

if duplicate_ids != 0:
    raise ValueError("STOP — duplicate session IDs.")

if unique_ids != 11880:
    raise ValueError(
        f"STOP — expected 11880 unique sessions, got {unique_ids}"
    )

print("Session integrity: PASS")


# ----------------------------------------------------------------
# 4. LABEL LEAKAGE
# ----------------------------------------------------------------

FORBIDDEN_LABELS = {
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique",
}

source_forbidden = sorted(
    set(source_df.columns).intersection(FORBIDDEN_LABELS)
)

print("\n[4] LABEL LEAKAGE")
print("Forbidden label columns in ML input:", source_forbidden)

if source_forbidden:
    raise ValueError(
        "STOP — forbidden label columns detected: "
        + ", ".join(source_forbidden)
    )

print("Label leakage: NO")


# ----------------------------------------------------------------
# 5. ASSIGNMENT SESSION-ID DETECTION
# ----------------------------------------------------------------

def detect_session_column(df):
    candidates = [
        "reconstructed_session_id",
        "session_id",
        "raw_session_id",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(
        "STOP — could not identify session ID column."
    )


def detect_cluster_column(df):
    candidates = [
        "cluster",
        "cluster_label",
        "kmeans_cluster",
        "dbscan_cluster",
        "agglomerative_cluster",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    # fallback: find a likely non-ID column
    non_id = [
        c for c in df.columns
        if c not in {
            "reconstructed_session_id",
            "session_id",
            "raw_session_id",
        }
    ]

    if len(non_id) == 1:
        return non_id[0]

    raise ValueError(
        "STOP — could not identify cluster label column."
    )


# ----------------------------------------------------------------
# 6. ASSIGNMENT INTEGRITY
# ----------------------------------------------------------------

assignment_results = {}

for model_name, df in [
    ("KMeans", kmeans_assign),
    ("DBSCAN", dbscan_assign),
    ("Agglomerative", agg_assign),
]:

    sid_col = detect_session_column(df)
    cluster_col = detect_cluster_column(df)

    missing = int(df[sid_col].isna().sum())
    duplicate = int(df[sid_col].duplicated().sum())
    unique = int(df[sid_col].nunique())
    missing_clusters = int(df[cluster_col].isna().sum())

    source_ids = set(source_df[SESSION_ID].astype(str))
    model_ids = set(df[sid_col].astype(str))

    ids_match = source_ids == model_ids

    print(f"\n[5] {model_name} ASSIGNMENT INTEGRITY")
    print("Rows:", len(df))
    print("Missing IDs:", missing)
    print("Duplicate IDs:", duplicate)
    print("Unique IDs:", unique)
    print("Missing cluster labels:", missing_clusters)
    print("Session IDs match source:", ids_match)

    if missing != 0:
        raise ValueError(f"STOP — {model_name} has missing session IDs.")

    if duplicate != 0:
        raise ValueError(f"STOP — {model_name} has duplicate session IDs.")

    if unique != 11880:
        raise ValueError(
            f"STOP — {model_name} does not contain 11880 sessions."
        )

    if missing_clusters != 0:
        raise ValueError(
            f"STOP — {model_name} has missing cluster assignments."
        )

    if not ids_match:
        raise ValueError(
            f"STOP — {model_name} session IDs do not match source."
        )

    assignment_results[model_name] = {
        "rows": len(df),
        "missing_ids": missing,
        "duplicate_ids": duplicate,
        "unique_ids": unique,
        "missing_cluster_labels": missing_clusters,
        "session_ids_match_source": bool(ids_match),
        "cluster_column": cluster_col,
    }

    print("Assignment integrity: PASS")


# ----------------------------------------------------------------
# 7. MODEL COMPARISON
# ----------------------------------------------------------------

print("\n[6] MODEL COMPARISON")

comparison_rows = []

# ---- KMeans ----
if "silhouette_score" not in kmeans_comp.columns:
    raise ValueError("KMeans comparison missing silhouette_score.")

km_best = kmeans_comp.loc[
    kmeans_comp["silhouette_score"].idxmax()
].copy()

comparison_rows.append({
    "model": "KMeans",
    "selected_parameter": f"k={int(km_best['k'])}",
    "clusters_found": int(km_best["clusters_found"]),
    "noise_count": 0,
    "noise_fraction": 0.0,
    "silhouette_score": float(km_best["silhouette_score"]),
    "davies_bouldin_score": float(km_best["davies_bouldin_score"]),
    "calinski_harabasz_score": float(
        km_best["calinski_harabasz_score"]
    ),
})

# ---- DBSCAN ----
db_best = dbscan_comp.loc[
    dbscan_comp["silhouette_score"].idxmax()
].copy()

comparison_rows.append({
    "model": "DBSCAN",
    "selected_parameter": (
        f"eps={db_best['eps']}, "
        f"min_samples={int(db_best['min_samples'])}"
    ),
    "clusters_found": int(db_best["clusters_found"]),
    "noise_count": int(db_best["noise_count"]),
    "noise_fraction": float(db_best["noise_fraction"]),
    "silhouette_score": float(db_best["silhouette_score"]),
    "davies_bouldin_score": float(db_best["davies_bouldin_score"]),
    "calinski_harabasz_score": float(
        db_best["calinski_harabasz_score"]
    ),
})

# ---- Agglomerative ----
agg_best = agg_comp.loc[
    agg_comp["silhouette_score"].idxmax()
].copy()

comparison_rows.append({
    "model": "Agglomerative",
    "selected_parameter": f"k={int(agg_best['k'])}",
    "clusters_found": int(agg_best["clusters_found"]),
    "noise_count": 0,
    "noise_fraction": 0.0,
    "silhouette_score": float(agg_best["silhouette_score"]),
    "davies_bouldin_score": float(agg_best["davies_bouldin_score"]),
    "calinski_harabasz_score": float(
        agg_best["calinski_harabasz_score"]
    ),
})

comparison_df = pd.DataFrame(comparison_rows)

print(comparison_df.to_string(index=False))


# ----------------------------------------------------------------
# 8. METRIC SANITY
# ----------------------------------------------------------------

print("\n[7] METRIC SANITY")

metric_cols = [
    "silhouette_score",
    "davies_bouldin_score",
    "calinski_harabasz_score",
]

if not np.isfinite(
    comparison_df[metric_cols].to_numpy(dtype=float)
).all():
    raise ValueError(
        "STOP — non-finite model comparison metrics detected."
    )

print("Comparison metrics finite: PASS")


# ----------------------------------------------------------------
# 9. PRIMARY RANKING
# ----------------------------------------------------------------
# Primary criterion:
#   highest Silhouette
#
# Secondary reporting:
#   lower Davies-Bouldin is better
#   higher Calinski-Harabasz is better
#
# This does NOT blindly declare the final production model.
# It records the metric-based ranking for Step 6F.

comparison_df["silhouette_rank"] = (
    comparison_df["silhouette_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

comparison_df["davies_bouldin_rank"] = (
    comparison_df["davies_bouldin_score"]
    .rank(ascending=True, method="min")
    .astype(int)
)

comparison_df["calinski_harabasz_rank"] = (
    comparison_df["calinski_harabasz_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

comparison_df = comparison_df.sort_values(
    by=[
        "silhouette_score",
        "davies_bouldin_score",
    ],
    ascending=[
        False,
        True,
    ],
).reset_index(drop=True)

comparison_df["overall_metric_rank"] = (
    np.arange(1, len(comparison_df) + 1)
)

print("\n[8] PRIMARY MODEL RANKING")
print(
    comparison_df[
        [
            "overall_metric_rank",
            "model",
            "selected_parameter",
            "silhouette_score",
            "davies_bouldin_score",
            "calinski_harabasz_score",
            "clusters_found",
            "noise_fraction",
        ]
    ].to_string(index=False)
)


# ----------------------------------------------------------------
# 10. SAVE FINAL COMPARISON
# ----------------------------------------------------------------

comparison_df.to_csv(
    FINAL_COMPARISON,
    index=False
)

comparison_sha = sha256_file(FINAL_COMPARISON)


# ----------------------------------------------------------------
# 11. AUDIT RECORD
# ----------------------------------------------------------------

audit = {
    "step": "6F",
    "title": "Unsupervised ML Audit + Model Comparison",

    "source": str(SOURCE),
    "source_sha256": actual_source_sha,

    "dataset": {
        "rows": int(len(source_df)),
        "columns": int(len(source_df.columns)),
        "unique_sessions": unique_ids,
        "missing_session_ids": missing_ids,
        "duplicate_session_ids": duplicate_ids,
    },

    "label_leakage": {
        "forbidden_columns": sorted(FORBIDDEN_LABELS),
        "detected_forbidden_columns": source_forbidden,
        "leakage_detected": bool(source_forbidden),
    },

    "assignment_integrity": assignment_results,

    "model_comparison": comparison_df.to_dict(
        orient="records"
    ),

    "primary_selection_criterion": (
        "Highest Silhouette score; "
        "Davies-Bouldin and Calinski-Harabasz "
        "reported as secondary metrics."
    ),

    "outputs": {
        "comparison_csv": str(FINAL_COMPARISON),
        "comparison_sha256": comparison_sha,
    },

    "source_protection": {
        "source_modified": False
    },

    "status": "PASS",
}

with open(FINAL_AUDIT, "w", encoding="utf-8") as f:
    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False
    )

audit_sha = sha256_file(FINAL_AUDIT)


# ----------------------------------------------------------------
# 12. SHA MANIFEST
# ----------------------------------------------------------------

manifest = {
    "step": "6F",
    "files": {
        "source": {
            "path": str(SOURCE),
            "sha256": actual_source_sha,
        },
        "model_comparison": {
            "path": str(FINAL_COMPARISON),
            "sha256": comparison_sha,
        },
        "audit": {
            "path": str(FINAL_AUDIT),
            "sha256": audit_sha,
        },
    }
}

with open(FINAL_MANIFEST, "w", encoding="utf-8") as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# ----------------------------------------------------------------
# 13. FINAL CHECKS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 6F — COMPLETE")
print("=" * 70)

print("\nSource:")
print(SOURCE)

print("\nSource SHA:")
print(actual_source_sha)

print("\nFinal comparison:")
print(FINAL_COMPARISON)

print("\nComparison SHA:")
print(comparison_sha)

print("\nAudit:")
print(FINAL_AUDIT)

print("\nAudit SHA:")
print(audit_sha)

print("\nManifest:")
print(FINAL_MANIFEST)

print("\nSafety:")
print("Source modified:          NO")
print("Sessions preserved:       YES")
print("Session IDs preserved:    YES")
print("K-Means assignments:      COMPLETE")
print("DBSCAN assignments:       COMPLETE")
print("Agglomerative assignments: COMPLETE")
print("Labels used:              NO")
print("Label leakage:            NO")
print("Non-finite comparison metrics: NO")

print("\nSTATUS: PASS")
print("NEXT: STEP 7 — CLUSTER PROFILING / INTERPRETATION")

STEP 6F — UNSUPERVISED ML AUDIT + MODEL COMPARISON

[1] SOURCE SHA-256
Expected: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual:   bf38b32554f91ff24b334415930656cee7d7b1e3ecf93dd584fbfcb059a050a4


ValueError: STOP — source SHA mismatch.

In [ ]:
# ================================================================
# STEP 6F — UNSUPERVISED ML AUDIT + MODEL COMPARISON
# CORRECTED INPUT: STEP 6B ML-READY DATASET
# ================================================================

import os
import json
import hashlib
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/content/step6_unsupervised_ml")

# ------------------------------------------------
# CORRECT STEP 6F INPUT
# ------------------------------------------------

SOURCE = BASE / "step6b_preprocessing" / \
    "session_behavioral_features_v4_ml_ready.csv"

EXPECTED_SOURCE_SHA = (
    "bf38b32554f91ff24b334415930656cee7d7b1e3ecf93dd584fbfcb059a050a4"
)

# ------------------------------------------------
# MODEL OUTPUTS
# ------------------------------------------------

KMEANS_ASSIGNMENTS = BASE / "step6c_kmeans" / \
    "kmeans_cluster_assignments.csv"

KMEANS_COMPARISON = BASE / "step6c_kmeans" / \
    "kmeans_model_comparison.csv"

DBSCAN_ASSIGNMENTS = BASE / "step6d_dbscan" / \
    "dbscan_cluster_assignments.csv"

DBSCAN_COMPARISON = BASE / "step6d_dbscan" / \
    "dbscan_model_comparison.csv"

AGG_ASSIGNMENTS = BASE / "step6e_agglomerative" / \
    "agglomerative_cluster_assignments.csv"

AGG_COMPARISON = BASE / "step6e_agglomerative" / \
    "agglomerative_model_comparison.csv"

OUTDIR = BASE / "step6f_final_audit"
OUTDIR.mkdir(parents=True, exist_ok=True)

FINAL_COMPARISON = OUTDIR / "step6f_model_comparison.csv"
FINAL_AUDIT = OUTDIR / "step6f_unsupervised_ml_audit.json"
FINAL_MANIFEST = OUTDIR / "step6f_manifest.json"


# ------------------------------------------------
# SHA-256
# ------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------
# REQUIRED FILES
# ------------------------------------------------

required_files = {
    "ml_ready_source": SOURCE,
    "kmeans_assignments": KMEANS_ASSIGNMENTS,
    "kmeans_comparison": KMEANS_COMPARISON,
    "dbscan_assignments": DBSCAN_ASSIGNMENTS,
    "dbscan_comparison": DBSCAN_COMPARISON,
    "agglomerative_assignments": AGG_ASSIGNMENTS,
    "agglomerative_comparison": AGG_COMPARISON,
}

missing = [
    name
    for name, path in required_files.items()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "STOP — required Step 6 files are missing: "
        + ", ".join(missing)
    )


print("=" * 70)
print("STEP 6F — UNSUPERVISED ML AUDIT + MODEL COMPARISON")
print("=" * 70)


# ------------------------------------------------
# 1. SOURCE SHA
# ------------------------------------------------

actual_source_sha = sha256_file(SOURCE)

print("\n[1] SOURCE SHA-256")
print("Expected:", EXPECTED_SOURCE_SHA)
print("Actual:  ", actual_source_sha)

if actual_source_sha != EXPECTED_SOURCE_SHA:
    raise ValueError(
        "STOP — ML-ready source SHA mismatch."
    )

print("Verification: PASS")


# ------------------------------------------------
# 2. LOAD ML-READY DATA
# ------------------------------------------------

source_df = pd.read_csv(SOURCE)

kmeans_assign = pd.read_csv(KMEANS_ASSIGNMENTS)
dbscan_assign = pd.read_csv(DBSCAN_ASSIGNMENTS)
agg_assign = pd.read_csv(AGG_ASSIGNMENTS)

kmeans_comp = pd.read_csv(KMEANS_COMPARISON)
dbscan_comp = pd.read_csv(DBSCAN_COMPARISON)
agg_comp = pd.read_csv(AGG_COMPARISON)

print("\n[2] ML-READY DATASET")
print("Shape:", source_df.shape)
print("Rows:", len(source_df))
print("Columns:", len(source_df.columns))


# ------------------------------------------------
# 3. SESSION ID
# ------------------------------------------------

SESSION_ID = "reconstructed_session_id"

if SESSION_ID not in source_df.columns:
    raise ValueError(
        "STOP — reconstructed_session_id missing."
    )

missing_ids = int(
    source_df[SESSION_ID].isna().sum()
)

duplicate_ids = int(
    source_df[SESSION_ID].duplicated().sum()
)

unique_ids = int(
    source_df[SESSION_ID].nunique()
)

print("\n[3] SESSION INTEGRITY")
print("Missing IDs:", missing_ids)
print("Duplicate IDs:", duplicate_ids)
print("Unique IDs:", unique_ids)

if missing_ids != 0:
    raise ValueError("STOP — missing session IDs.")

if duplicate_ids != 0:
    raise ValueError("STOP — duplicate session IDs.")

if unique_ids != 11880:
    raise ValueError(
        f"STOP — expected 11880 sessions, got {unique_ids}"
    )

print("Session integrity: PASS")


# ------------------------------------------------
# 4. LABEL LEAKAGE
# ------------------------------------------------

FORBIDDEN_LABELS = {
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique",
}

detected_labels = sorted(
    set(source_df.columns).intersection(
        FORBIDDEN_LABELS
    )
)

print("\n[4] LABEL LEAKAGE")
print("Forbidden label columns:", detected_labels)

if detected_labels:
    raise ValueError(
        "STOP — label leakage detected: "
        + ", ".join(detected_labels)
    )

print("Label leakage: NO")


# ------------------------------------------------
# 5. VERIFY ASSIGNMENTS
# ------------------------------------------------

def detect_session_column(df):

    candidates = [
        "reconstructed_session_id",
        "session_id",
        "raw_session_id",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(
        "STOP — session ID column not found."
    )


def detect_cluster_column(df):

    candidates = [
        "cluster",
        "cluster_label",
        "kmeans_cluster",
        "dbscan_cluster",
        "agglomerative_cluster",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    non_id = [
        c for c in df.columns
        if c not in {
            "reconstructed_session_id",
            "session_id",
            "raw_session_id",
        }
    ]

    if len(non_id) == 1:
        return non_id[0]

    raise ValueError(
        "STOP — cluster column could not be detected."
    )


source_ids = set(
    source_df[SESSION_ID].astype(str)
)

assignment_audit = {}

for model_name, df in [
    ("KMeans", kmeans_assign),
    ("DBSCAN", dbscan_assign),
    ("Agglomerative", agg_assign),
]:

    sid_col = detect_session_column(df)
    cluster_col = detect_cluster_column(df)

    missing = int(df[sid_col].isna().sum())
    duplicate = int(df[sid_col].duplicated().sum())
    unique = int(df[sid_col].nunique())
    missing_clusters = int(
        df[cluster_col].isna().sum()
    )

    model_ids = set(
        df[sid_col].astype(str)
    )

    ids_match = (
        source_ids == model_ids
    )

    print(
        f"\n[5] {model_name} ASSIGNMENT INTEGRITY"
    )

    print("Rows:", len(df))
    print("Missing IDs:", missing)
    print("Duplicate IDs:", duplicate)
    print("Unique IDs:", unique)
    print(
        "Missing cluster labels:",
        missing_clusters
    )
    print(
        "Session IDs match source:",
        ids_match
    )

    if len(df) != 11880:
        raise ValueError(
            f"STOP — {model_name} row count mismatch."
        )

    if missing != 0:
        raise ValueError(
            f"STOP — {model_name} missing IDs."
        )

    if duplicate != 0:
        raise ValueError(
            f"STOP — {model_name} duplicate IDs."
        )

    if unique != 11880:
        raise ValueError(
            f"STOP — {model_name} unique ID mismatch."
        )

    if missing_clusters != 0:
        raise ValueError(
            f"STOP — {model_name} missing cluster labels."
        )

    if not ids_match:
        raise ValueError(
            f"STOP — {model_name} IDs do not match source."
        )

    assignment_audit[model_name] = {
        "rows": len(df),
        "missing_ids": missing,
        "duplicate_ids": duplicate,
        "unique_ids": unique,
        "missing_cluster_labels":
            missing_clusters,
        "session_ids_match_source":
            bool(ids_match),
        "cluster_column": cluster_col,
    }

    print("Assignment integrity: PASS")


# ------------------------------------------------
# 6. MODEL COMPARISON
# ------------------------------------------------

print("\n[6] MODEL COMPARISON")

rows = []

# K-MEANS
km_best = kmeans_comp.loc[
    kmeans_comp["silhouette_score"].idxmax()
]

rows.append({
    "model": "KMeans",
    "selected_parameter":
        f"k={int(km_best['k'])}",
    "clusters_found":
        int(km_best["clusters_found"]),
    "noise_count": 0,
    "noise_fraction": 0.0,
    "silhouette_score":
        float(km_best["silhouette_score"]),
    "davies_bouldin_score":
        float(km_best["davies_bouldin_score"]),
    "calinski_harabasz_score":
        float(km_best["calinski_harabasz_score"]),
})


# DBSCAN
db_best = dbscan_comp.loc[
    dbscan_comp["silhouette_score"].idxmax()
]

rows.append({
    "model": "DBSCAN",
    "selected_parameter":
        (
            f"eps={db_best['eps']}, "
            f"min_samples="
            f"{int(db_best['min_samples'])}"
        ),
    "clusters_found":
        int(db_best["clusters_found"]),
    "noise_count":
        int(db_best["noise_count"]),
    "noise_fraction":
        float(db_best["noise_fraction"]),
    "silhouette_score":
        float(db_best["silhouette_score"]),
    "davies_bouldin_score":
        float(db_best["davies_bouldin_score"]),
    "calinski_harabasz_score":
        float(db_best["calinski_harabasz_score"]),
})


# AGGLOMERATIVE
agg_best = agg_comp.loc[
    agg_comp["silhouette_score"].idxmax()
]

rows.append({
    "model": "Agglomerative",
    "selected_parameter":
        f"k={int(agg_best['k'])}",
    "clusters_found":
        int(agg_best["clusters_found"]),
    "noise_count": 0,
    "noise_fraction": 0.0,
    "silhouette_score":
        float(agg_best["silhouette_score"]),
    "davies_bouldin_score":
        float(agg_best["davies_bouldin_score"]),
    "calinski_harabasz_score":
        float(
            agg_best["calinski_harabasz_score"]
        ),
})


comparison_df = pd.DataFrame(rows)

print(
    comparison_df.to_string(index=False)
)


# ------------------------------------------------
# 7. METRIC SANITY
# ------------------------------------------------

metric_cols = [
    "silhouette_score",
    "davies_bouldin_score",
    "calinski_harabasz_score",
]

if not np.isfinite(
    comparison_df[metric_cols]
    .to_numpy(dtype=float)
).all():

    raise ValueError(
        "STOP — non-finite metrics detected."
    )

print("\n[7] METRIC SANITY")
print("Comparison metrics finite: PASS")


# ------------------------------------------------
# 8. RANKING
# ------------------------------------------------

comparison_df[
    "silhouette_rank"
] = comparison_df[
    "silhouette_score"
].rank(
    ascending=False,
    method="min"
).astype(int)

comparison_df[
    "davies_bouldin_rank"
] = comparison_df[
    "davies_bouldin_score"
].rank(
    ascending=True,
    method="min"
).astype(int)

comparison_df[
    "calinski_harabasz_rank"
] = comparison_df[
    "calinski_harabasz_score"
].rank(
    ascending=False,
    method="min"
).astype(int)

comparison_df = comparison_df.sort_values(
    by=[
        "silhouette_score",
        "davies_bouldin_score"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)

comparison_df[
    "overall_metric_rank"
] = np.arange(
    1,
    len(comparison_df) + 1
)

print("\n[8] MODEL RANKING")

print(
    comparison_df[
        [
            "overall_metric_rank",
            "model",
            "selected_parameter",
            "silhouette_score",
            "davies_bouldin_score",
            "calinski_harabasz_score",
            "clusters_found",
            "noise_fraction",
        ]
    ].to_string(index=False)
)


# ------------------------------------------------
# 9. SAVE COMPARISON
# ------------------------------------------------

comparison_df.to_csv(
    FINAL_COMPARISON,
    index=False
)

comparison_sha = sha256_file(
    FINAL_COMPARISON
)


# ------------------------------------------------
# 10. AUDIT JSON
# ------------------------------------------------

audit = {

    "step": "6F",

    "title":
        "Unsupervised ML Audit + Model Comparison",

    "source": str(SOURCE),

    "source_sha256":
        actual_source_sha,

    "dataset": {
        "rows": int(len(source_df)),
        "columns":
            int(len(source_df.columns)),
        "unique_sessions":
            unique_ids,
        "missing_session_ids":
            missing_ids,
        "duplicate_session_ids":
            duplicate_ids,
    },

    "label_leakage": {
        "forbidden_columns":
            sorted(FORBIDDEN_LABELS),
        "detected_forbidden_columns":
            detected_labels,
        "leakage_detected":
            bool(detected_labels),
    },

    "assignment_integrity":
        assignment_audit,

    "model_comparison":
        comparison_df.to_dict(
            orient="records"
        ),

    "selection_criterion":
        "Highest Silhouette score; "
        "Davies-Bouldin and "
        "Calinski-Harabasz reported "
        "as secondary metrics.",

    "source_protection": {
        "source_modified": False
    },

    "status": "PASS",
}


with open(
    FINAL_AUDIT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False
    )


audit_sha = sha256_file(
    FINAL_AUDIT
)


# ------------------------------------------------
# 11. MANIFEST
# ------------------------------------------------

manifest = {

    "step": "6F",

    "files": {

        "source": {
            "path": str(SOURCE),
            "sha256":
                actual_source_sha,
        },

        "model_comparison": {
            "path":
                str(FINAL_COMPARISON),
            "sha256":
                comparison_sha,
        },

        "audit": {
            "path":
                str(FINAL_AUDIT),
            "sha256":
                audit_sha,
        },
    }
}


with open(
    FINAL_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------
# 12. FINAL OUTPUT
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 6F — COMPLETE")
print("=" * 70)

print("\nSource:")
print(SOURCE)

print("\nSource SHA:")
print(actual_source_sha)

print("\nModel comparison:")
print(FINAL_COMPARISON)

print("\nComparison SHA:")
print(comparison_sha)

print("\nAudit:")
print(FINAL_AUDIT)

print("\nAudit SHA:")
print(audit_sha)

print("\nManifest:")
print(FINAL_MANIFEST)

print("\nSafety:")
print("Source modified:           NO")
print("Sessions preserved:        YES")
print("Session IDs preserved:     YES")
print("K-Means assignments:       COMPLETE")
print("DBSCAN assignments:        COMPLETE")
print("Agglomerative assignments: COMPLETE")
print("Labels used:               NO")
print("Label leakage:             NO")
print("Non-finite metrics:        NO")

print("\nSTATUS: PASS")
print("NEXT: STEP 7 — CLUSTER PROFILING / INTERPRETATION")

STEP 6F — UNSUPERVISED ML AUDIT + MODEL COMPARISON

[1] SOURCE SHA-256
Expected: bf38b32554f91ff24b334415930656cee7d7b1e3ecf93dd584fbfcb059a050a4
Actual:   bf38b32554f91ff24b334415930656cee7d7b1e3ecf93dd584fbfcb059a050a4
Verification: PASS

[2] ML-READY DATASET
Shape: (11880, 49)
Rows: 11880
Columns: 49

[3] SESSION INTEGRITY
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Session integrity: PASS

[4] LABEL LEAKAGE
Forbidden label columns: []
Label leakage: NO

[5] KMeans ASSIGNMENT INTEGRITY
Rows: 11880
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Missing cluster labels: 0
Session IDs match source: True
Assignment integrity: PASS

[5] DBSCAN ASSIGNMENT INTEGRITY
Rows: 11880
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Missing cluster labels: 0
Session IDs match source: True
Assignment integrity: PASS

[5] Agglomerative ASSIGNMENT INTEGRITY
Rows: 11880
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Missing cluster labels: 0
Session IDs match source: True
Assignment integ

In [ ]:
# ================================================================
# STEP 6 — COMPLETE ARTIFACT PACKAGE
# Create a single ZIP containing all important STEP 6 outputs
# ================================================================

import os
import zipfile
import hashlib

# ------------------------------------------------
# 1. Output package
# ------------------------------------------------
ZIP_PATH = "/content/STEP_6_UNSUPERVISED_ML_COMPLETE.zip"

# ------------------------------------------------
# 2. STEP 6 directories
# ------------------------------------------------
BASE = "/content/step6_unsupervised_ml"

FILES = [
    # -----------------------------
    # STEP 6B — Preprocessing
    # -----------------------------
    (
        f"{BASE}/step6b_preprocessing/session_behavioral_features_v4_ml_ready.csv",
        "step6b_preprocessing/session_behavioral_features_v4_ml_ready.csv"
    ),
    (
        f"{BASE}/step6b_preprocessing/session_behavioral_features_v4_ml_ready.sha256",
        "step6b_preprocessing/session_behavioral_features_v4_ml_ready.sha256"
    ),
    (
        f"{BASE}/step6b_preprocessing/standard_scaler_params.json",
        "step6b_preprocessing/standard_scaler_params.json"
    ),
    (
        f"{BASE}/step6b_preprocessing/step6b_preprocessing_audit.json",
        "step6b_preprocessing/step6b_preprocessing_audit.json"
    ),

    # -----------------------------
    # STEP 6C — K-Means
    # -----------------------------
    (
        f"{BASE}/step6c_kmeans/kmeans_model_comparison.csv",
        "step6c_kmeans/kmeans_model_comparison.csv"
    ),
    (
        f"{BASE}/step6c_kmeans/kmeans_cluster_assignments.csv",
        "step6c_kmeans/kmeans_cluster_assignments.csv"
    ),
    (
        f"{BASE}/step6c_kmeans/kmeans_cluster_assignments.sha256",
        "step6c_kmeans/kmeans_cluster_assignments.sha256"
    ),
    (
        f"{BASE}/step6c_kmeans/step6c_kmeans_audit.json",
        "step6c_kmeans/step6c_kmeans_audit.json"
    ),

    # -----------------------------
    # STEP 6D — DBSCAN
    # -----------------------------
    (
        f"{BASE}/step6d_dbscan/dbscan_model_comparison.csv",
        "step6d_dbscan/dbscan_model_comparison.csv"
    ),
    (
        f"{BASE}/step6d_dbscan/dbscan_cluster_assignments.csv",
        "step6d_dbscan/dbscan_cluster_assignments.csv"
    ),
    (
        f"{BASE}/step6d_dbscan/dbscan_cluster_assignments.sha256",
        "step6d_dbscan/dbscan_cluster_assignments.sha256"
    ),
    (
        f"{BASE}/step6d_dbscan/step6d_dbscan_audit.json",
        "step6d_dbscan/step6d_dbscan_audit.json"
    ),

    # -----------------------------
    # STEP 6E — Agglomerative
    # -----------------------------
    (
        f"{BASE}/step6e_agglomerative/agglomerative_model_comparison.csv",
        "step6e_agglomerative/agglomerative_model_comparison.csv"
    ),
    (
        f"{BASE}/step6e_agglomerative/agglomerative_cluster_assignments.csv",
        "step6e_agglomerative/agglomerative_cluster_assignments.csv"
    ),
    (
        f"{BASE}/step6e_agglomerative/agglomerative_cluster_assignments.sha256",
        "step6e_agglomerative/agglomerative_cluster_assignments.sha256"
    ),
    (
        f"{BASE}/step6e_agglomerative/step6e_agglomerative_audit.json",
        "step6e_agglomerative/step6e_agglomerative_audit.json"
    ),

    # -----------------------------
    # STEP 6F — Final Audit
    # -----------------------------
    (
        f"{BASE}/step6f_final_audit/step6f_model_comparison.csv",
        "step6f_final_audit/step6f_model_comparison.csv"
    ),
    (
        f"{BASE}/step6f_final_audit/step6f_unsupervised_ml_audit.json",
        "step6f_final_audit/step6f_unsupervised_ml_audit.json"
    ),
    (
        f"{BASE}/step6f_final_audit/step6f_manifest.json",
        "step6f_final_audit/step6f_manifest.json"
    ),
]

# ------------------------------------------------
# 3. Verify files before creating ZIP
# ------------------------------------------------
print("=" * 70)
print("STEP 6 — ARTIFACT PACKAGE VERIFICATION")
print("=" * 70)

missing = []
present = []

for source_path, archive_path in FILES:
    if os.path.isfile(source_path):
        size = os.path.getsize(source_path)
        present.append((source_path, archive_path, size))
        print(f"PASS: {archive_path} ({size:,} bytes)")
    else:
        missing.append(source_path)
        print(f"MISSING: {source_path}")

print()

if missing:
    print("STATUS: STOP")
    print("Missing files:")
    for f in missing:
        print(" -", f)

    raise FileNotFoundError(
        f"STEP 6 package incomplete. Missing {len(missing)} file(s)."
    )

print(f"All required files found: {len(present)}")
print()

# ------------------------------------------------
# 4. Create ZIP
# ------------------------------------------------
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for source_path, archive_path, _ in present:
        zf.write(source_path, arcname=archive_path)

# ------------------------------------------------
# 5. Verify ZIP contents
# ------------------------------------------------
print("=" * 70)
print("ZIP CONTENT VERIFICATION")
print("=" * 70)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = set(zf.namelist())

expected_names = {archive_path for _, archive_path, _ in present}

missing_from_zip = expected_names - names

if missing_from_zip:
    raise RuntimeError(
        "STOP — ZIP verification failed. "
        f"Missing entries: {sorted(missing_from_zip)}"
    )

print(f"Files packaged: {len(names)}")
print("ZIP contents: PASS")

# ------------------------------------------------
# 6. ZIP integrity test
# ------------------------------------------------
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    bad_file = zf.testzip()

if bad_file is not None:
    raise RuntimeError(
        f"STOP — ZIP integrity check failed: {bad_file}"
    )

print("ZIP integrity: PASS")

# ------------------------------------------------
# 7. SHA-256 of ZIP itself
# ------------------------------------------------
sha256 = hashlib.sha256()

with open(ZIP_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

ZIP_SHA = sha256.hexdigest()

ZIP_SIZE = os.path.getsize(ZIP_PATH)

print()
print("=" * 70)
print("STEP 6 — COMPLETE PACKAGE")
print("=" * 70)

print(f"ZIP: {ZIP_PATH}")
print(f"ZIP size: {ZIP_SIZE:,} bytes")
print(f"ZIP SHA-256: {ZIP_SHA}")
print(f"Artifacts: {len(names)}")
print()
print("STATUS: PASS")
print("READY TO DOWNLOAD")

STEP 6 — ARTIFACT PACKAGE VERIFICATION
PASS: step6b_preprocessing/session_behavioral_features_v4_ml_ready.csv (10,922,243 bytes)
PASS: step6b_preprocessing/session_behavioral_features_v4_ml_ready.sha256 (110 bytes)
PASS: step6b_preprocessing/standard_scaler_params.json (8,230 bytes)
PASS: step6b_preprocessing/step6b_preprocessing_audit.json (1,153 bytes)
PASS: step6c_kmeans/kmeans_model_comparison.csv (795 bytes)
PASS: step6c_kmeans/kmeans_cluster_assignments.csv (343,890 bytes)
PASS: step6c_kmeans/kmeans_cluster_assignments.sha256 (97 bytes)
PASS: step6c_kmeans/step6c_kmeans_audit.json (1,062 bytes)
PASS: step6d_dbscan/dbscan_model_comparison.csv (2,279 bytes)
PASS: step6d_dbscan/dbscan_cluster_assignments.csv (350,720 bytes)
PASS: step6d_dbscan/dbscan_cluster_assignments.sha256 (97 bytes)
PASS: step6d_dbscan/step6d_dbscan_audit.json (3,331 bytes)
PASS: step6e_agglomerative/agglomerative_model_comparison.csv (624 bytes)
PASS: step6e_agglomerative/agglomerative_cluster_assignments.csv 

In [ ]:
from google.colab import files

ZIP_PATH = "/content/STEP_6_UNSUPERVISED_ML_COMPLETE.zip"

files.download(ZIP_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>